In [21]:
# ============================================================================
# CELL 1: ERG PIPELINE CONFIGURATION (ISCEV 2022 Compliant)
# Single source of truth - Version 1.0.0
# ============================================================================

import numpy as np
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from datetime import datetime

@dataclass
class ERGConfig:
    """Master configuration for ISCEV 2022-compliant ERG processing pipeline"""

    # ========================================================================
    # ISCEV 2022 MANDATED PARAMETERS (Page 7, Col 2)
    # ========================================================================
    ISCEV_HIGHPASS_HZ: float = 0.3          # ≤0.3 Hz (Page 7, Col 2, Para 3)
    ISCEV_LOWPASS_HZ: float = 300.0         # ≥300 Hz (Page 7, Col 2, Para 3)
    ISCEV_MIN_SAMPLING_RATE_HZ: float = 1000.0   # ≥1 kHz (Page 7, Col 2, Para 2)
    ISCEV_MIN_PRESTIMULUS_MS: float = 20.0       # ≥20 ms (Page 7, Col 2, Para 5)
    ISCEV_MIN_RECORDING_MS: float = 300.0        # ≥300 ms (Page 7, Col 2, Para 5)
    ISCEV_OP_BAND_LOW_HZ: float = 75.0           # 75 Hz minimum (Page 7, Col 2, Para 4)
    ISCEV_FLASH_MAX_DURATION_MS: float = 5.0     # <5 ms (Page 4, Col 2, Para 2)

    # ========================================================================
    # FILTER PARAMETERS (Clinical Standard, not ISCEV-mandated)
    # ========================================================================
    BUTTERWORTH_ORDER: int = 4              # 4th-order (clinical standard)
    MEDIAN_KERNEL_MS: float = 5.0           # 5 ms for spike removal
    NOTCH_QUALITY_FACTOR: float = 30.0      # Q=30 (~1.67 Hz bandwidth)
    NOTCH_DEFAULT_STATE: bool = False       # OFF by default per ISCEV 2022

    # ========================================================================
    # ELECTRODE-SPECIFIC SNR THRESHOLDS (Chapter 2 Clinical Best Practice)
    # Not ISCEV-mandated - ISCEV provides no numerical SNR thresholds
    # ========================================================================
    ELECTRODE_SNR_THRESHOLDS: Dict[str, Dict[str, float]] = field(default_factory=lambda: {
        'contact_lens': {'pass': 8.0, 'warning_min': 4.0, 'fail_below': 4.0},
        'gold_foil': {'pass': 6.0, 'warning_min': 3.0, 'fail_below': 3.0},
        'dtl_fiber': {'pass': 4.0, 'warning_min': 2.5, 'fail_below': 2.5},
        'skin': {'pass': 3.0, 'warning_min': 1.5, 'fail_below': 1.5}
    })

    # ========================================================================
    # MAINS INTERFERENCE DETECTION (Audit only - NOT auto-apply)
    # ========================================================================
    MAINS_CANDIDATE_FREQS_HZ: List[float] = field(default_factory=lambda: [50.0, 60.0])
    MAINS_DETECTION_THRESHOLD_DB: float = 6.0   # Peak must exceed noise floor by 6 dB
    MAINS_DETECTION_BANDWIDTH_HZ: float = 2.0   # ±2 Hz around candidate

    # ========================================================================
    # OP BAND DETECTION
    # ========================================================================
    OP_BAND_SNR_THRESHOLD_DB: float = 3.0       # Minimum SNR to declare OP available
    OP_BAND_HIGH_HZ: float = 300.0              # Upper bound for OP band
    NOISE_BAND_LOW_HZ: float = 400.0            # For estimating noise floor
    NOISE_BAND_HIGH_HZ: float = 500.0           # For estimating noise floor

    # ========================================================================
    # QUALITY GRADING SCORES
    # ========================================================================
    QUALITY_SCORES: Dict[str, int] = field(default_factory=lambda: {
        'op_excellent': 3,      # OP SNR ≥10 dB
        'op_good': 2,           # OP available but SNR <10 dB
        'bandwidth_full': 2,    # ≥250 Hz
        'bandwidth_limited': 1, # 150-250 Hz
        'no_spikes': 2,         # spike_count = 0
        'minor_spikes': 1,      # spike_count ≤5
        'no_mains': 1           # No mains detected
    })

    # ========================================================================
    # QUALITY GRADE THRESHOLDS
    # ========================================================================
    QUALITY_GRADE_THRESHOLDS: Dict[str, int] = field(default_factory=lambda: {
        'A': 7,  # Score ≥7 = Excellent
        'B': 5,  # Score ≥5 = Good
        'C': 3   # Score ≥3 = Acceptable
    })

    # ========================================================================
    # AGE GROUP REFERENCE HANDLING (Committee Suggestion #3)
    # ========================================================================
    AGE_GROUPS: List[str] = field(default_factory=lambda: [
        '0-12mo', '1-5y', '6-12y', '13-17y', '18-80y', '80+y', 'unknown'
    ])
    DEFAULT_AGE_GROUP: str = '18-80y'

    # ========================================================================
    # PRE-STIMULUS BASELINE HANDLING (Committee Suggestion #1)
    # ========================================================================
    PRESTIMULUS_STATUS_THRESHOLDS_MS: Dict[str, float] = field(default_factory=lambda: {
        'adequate': 20.0,      # ≥20 ms = ISCEV compliant
        'partial_min': 10.0,   # 10-19 ms = partial warning
        'insufficient_min': 0.0 # <10 ms = insufficient
    })

    # ========================================================================
    # PIPELINE METADATA
    # ========================================================================
    PIPELINE_VERSION: str = "2.3.2"
    VERSION_DATE: str = "2026-05-18"
    PIPELINE_NAME: str = "ISCEV 2022-Compliant ERG Processing Pipeline"

    # ========================================================================
    # STFT SPECTROGRAM PARAMETERS (Chapter 8)
    # ========================================================================
    STFT_WINDOW: str = 'hamming'           # Default for deep learning
    STFT_NPERSEG: int = 64                 # 32 ms at 2000 Hz
    STFT_NOVERLAP: int = 56                # 87.5% overlap
    STFT_FMIN_HZ: float = 0.0              # Full ISCEV passband
    STFT_FMAX_HZ: float = 300.0            # ISCEV upper cutoff
    STFT_OUTPUT_H: int = 224               # ViT input height
    STFT_OUTPUT_W: int = 224               # ViT input width
    STFT_NORM: str = 'db_zscore'           # Z-score normalization


    def get_electrode_snr_status(self, electrode_type: str, snr_db: float) -> Tuple[str, str]:
        """Return (status, message) for given electrode type and SNR"""
        electrode_key = electrode_type.lower().replace(' ', '_')
        if electrode_key not in self.ELECTRODE_SNR_THRESHOLDS:
            return "UNKNOWN", f"Unknown electrode type: {electrode_type}"

        thresholds = self.ELECTRODE_SNR_THRESHOLDS[electrode_key]
        if snr_db >= thresholds['pass']:
            return "PASS", f"SNR {snr_db:.1f} dB ≥ {thresholds['pass']} dB threshold"
        elif snr_db >= thresholds['warning_min']:
            return "WARNING", f"SNR {snr_db:.1f} dB is below PASS threshold ({thresholds['pass']} dB)"
        else:
            return "FAIL", f"SNR {snr_db:.1f} dB is below WARNING threshold ({thresholds['warning_min']} dB)"

    def get_prestimulus_status(self, prestimulus_ms: float) -> Tuple[str, str]:
        """Return (status, message) for pre-stimulus baseline duration"""
        if prestimulus_ms >= self.PRESTIMULUS_STATUS_THRESHOLDS_MS['adequate']:
            return "ADEQUATE", f"Pre-stimulus baseline {prestimulus_ms:.1f} ms (≥20 ms ISCEV compliant)"
        elif prestimulus_ms >= self.PRESTIMULUS_STATUS_THRESHOLDS_MS['partial_min']:
            return "PARTIAL", f"Pre-stimulus baseline {prestimulus_ms:.1f} ms (<20 ms ISCEV minimum). Baseline estimate may be noisy."
        elif prestimulus_ms > 0:
            return "INSUFFICIENT", f"Pre-stimulus baseline {prestimulus_ms:.1f} ms (<10 ms). OP features may be unreliable."
        else:
            return "ABSENT", "No pre-stimulus baseline recorded. a-wave amplitude measurement may be inaccurate."

    def get_age_group_flag(self, age_group: str) -> Tuple[str, str]:
        """Return (reference_source, flag_message) for age group"""
        age_flags = {
            '0-12mo': ("pediatric", "WARNING: Pediatric reference ranges used; adult comparison not appropriate"),
            '1-5y': ("pediatric", "WARNING: Pediatric reference ranges used; adult comparison not appropriate"),
            '6-12y': ("pediatric_caution", "CAUTION: Partial maturation; interpret with caution"),
            '13-17y': ("near_adult", "Near-adult reference ranges applied"),
            '18-80y': ("adult", None),
            '80+y': ("elderly_caution", "CAUTION: Elderly reference ranges; age-related amplitude reduction expected"),
            'unknown': ("adult_fallback", "WARNING: Age not provided; adult reference ranges applied by default")
        }
        return age_flags.get(age_group, ("unknown", "WARNING: Unrecognized age group; adult ranges applied"))

    def to_dict(self) -> Dict:
        """Return configuration as dictionary for audit log"""
        return {
            'pipeline_version': self.PIPELINE_VERSION,
            'pipeline_name': self.PIPELINE_NAME,
            'iscev_highpass_hz': self.ISCEV_HIGHPASS_HZ,
            'iscev_lowpass_hz': self.ISCEV_LOWPASS_HZ,
            'iscev_min_sampling_rate_hz': self.ISCEV_MIN_SAMPLING_RATE_HZ,
            'iscev_min_prestimulus_ms': self.ISCEV_MIN_PRESTIMULUS_MS,
            'butterworth_order': self.BUTTERWORTH_ORDER,
            'median_kernel_ms': self.MEDIAN_KERNEL_MS,
            'notch_default_state': self.NOTCH_DEFAULT_STATE,
            'notch_q': self.NOTCH_QUALITY_FACTOR,
            'electrode_snr_thresholds': self.ELECTRODE_SNR_THRESHOLDS,
            'mains_detection_threshold_db': self.MAINS_DETECTION_THRESHOLD_DB,
            'op_band_snr_threshold_db': self.OP_BAND_SNR_THRESHOLD_DB,
            'quality_grade_thresholds': self.QUALITY_GRADE_THRESHOLDS
        }


# Instantiate master configuration
CONFIG = ERGConfig()

# Print confirmation
print("=" * 60)
print("ERG PIPELINE CONFIGURATION LOADED")
print("=" * 60)
print(f"Version: {CONFIG.PIPELINE_VERSION}")
print(f"ISCEV 2022 Compliance: High-pass {CONFIG.ISCEV_HIGHPASS_HZ} Hz, Low-pass {CONFIG.ISCEV_LOWPASS_HZ} Hz")
print(f"Notch filter default state: {'ON' if CONFIG.NOTCH_DEFAULT_STATE else 'OFF (ISCEV 2022 compliant)'}")
print(f"Electrode types supported: {list(CONFIG.ELECTRODE_SNR_THRESHOLDS.keys())}")
print(f"Age groups supported: {CONFIG.AGE_GROUPS}")
print("=" * 60)

ERG PIPELINE CONFIGURATION LOADED
Version: 2.3.2
ISCEV 2022 Compliance: High-pass 0.3 Hz, Low-pass 300.0 Hz
Notch filter default state: OFF (ISCEV 2022 compliant)
Electrode types supported: ['contact_lens', 'gold_foil', 'dtl_fiber', 'skin']
Age groups supported: ['0-12mo', '1-5y', '6-12y', '13-17y', '18-80y', '80+y', 'unknown']


In [22]:
# ============================================================================
# CELL 2: UTILITY FUNCTIONS (Fixed deprecation warnings - trapezoid not trapz)
# ============================================================================

import numpy as np
import pandas as pd
from scipy.signal import welch
from typing import Tuple, Dict, Any

def load_erg_csv(filepath: str) -> Tuple[np.ndarray, np.ndarray, float, Dict]:
    """Load ERG CSV file in standardized format (Time_ms, Amplitude_uV)"""
    df = pd.read_csv(filepath)

    time_col = None
    amp_col = None

    for col in df.columns:
        col_lower = col.lower()
        if 'time' in col_lower or 'ms' in col_lower:
            time_col = col
        if 'amplitude' in col_lower or 'uv' in col_lower or 'μv' in col_lower:
            amp_col = col

    if time_col is None or amp_col is None:
        raise ValueError(f"Cannot find time and amplitude columns. Found: {df.columns.tolist()}")

    time_ms = df[time_col].values
    signal_uv = df[amp_col].values

    dt_ms = np.median(np.diff(time_ms))
    fs_hz = 1000.0 / dt_ms

    metadata = {
        'filename': filepath,
        'n_samples': len(signal_uv),
        'duration_ms': time_ms[-1] - time_ms[0],
        'fs_hz': fs_hz,
        'time_col': time_col,
        'amp_col': amp_col
    }

    return time_ms, signal_uv, fs_hz, metadata


def compute_kernel_samples(fs_hz: float, kernel_ms: float = 5.0) -> int:
    """Convert kernel duration (ms) to samples, ensuring odd number for median filter"""
    kernel = int(fs_hz * kernel_ms / 1000)
    if kernel % 2 == 0:
        kernel += 1
    return max(kernel, 3)


def validate_iscev_requirements(fs_hz: float, prestimulus_ms: float, total_duration_ms: float) -> Dict[str, Any]:
    """Validate ISCEV 2022 minimum requirements"""
    post_stimulus_ms = total_duration_ms - prestimulus_ms

    results = {
        'sampling_rate_ok': fs_hz >= CONFIG.ISCEV_MIN_SAMPLING_RATE_HZ,
        'sampling_rate_value': fs_hz,
        'sampling_rate_required': CONFIG.ISCEV_MIN_SAMPLING_RATE_HZ,
        'prestimulus_ok': prestimulus_ms >= CONFIG.ISCEV_MIN_PRESTIMULUS_MS,
        'prestimulus_value': prestimulus_ms,
        'prestimulus_required': CONFIG.ISCEV_MIN_PRESTIMULUS_MS,
        'post_stimulus_ok': post_stimulus_ms >= CONFIG.ISCEV_MIN_RECORDING_MS,
        'post_stimulus_value': post_stimulus_ms,
        'post_stimulus_required': CONFIG.ISCEV_MIN_RECORDING_MS,
        'overall_compliant': False
    }

    results['overall_compliant'] = results['sampling_rate_ok'] and results['prestimulus_ok'] and results['post_stimulus_ok']

    warnings_list = []
    if not results['sampling_rate_ok']:
        warnings_list.append(f"Sampling rate {fs_hz:.1f} Hz below ISCEV minimum {CONFIG.ISCEV_MIN_SAMPLING_RATE_HZ} Hz")
    if not results['prestimulus_ok']:
        warnings_list.append(f"Pre-stimulus baseline {prestimulus_ms:.1f} ms below ISCEV minimum {CONFIG.ISCEV_MIN_PRESTIMULUS_MS} ms")
    if not results['post_stimulus_ok']:
        warnings_list.append(f"Post-stimulus duration {post_stimulus_ms:.0f} ms below ISCEV minimum {CONFIG.ISCEV_MIN_RECORDING_MS} ms")

    results['warnings'] = warnings_list
    return results


def calculate_snr(signal_uv: np.ndarray, fs_hz: float, prestimulus_samples: int = 0) -> float:
    """
    Calculate SNR using time-domain RMS per Chapter 2 §2.2.1.
    SNR = 20 * log10(post-stimulus RMS / pre-stimulus RMS)

    Args:
        signal_uv: Full signal in microvolts
        fs_hz: Sampling rate in Hz (unused but kept for API compatibility)
        prestimulus_samples: Number of samples before flash onset

    Returns:
        SNR in decibels (dB), or 0.0 if insufficient pre-stimulus samples
    """
    if prestimulus_samples < 5 or prestimulus_samples >= len(signal_uv):
        return 0.0

    # Noise RMS from pre-stimulus baseline
    noise_rms = float(np.sqrt(np.mean(signal_uv[:prestimulus_samples]**2)))

    # Signal RMS from post-stimulus window
    signal_rms = float(np.sqrt(np.mean(signal_uv[prestimulus_samples:]**2)))

    # SNR linear (amplitude ratio)
    snr_linear = signal_rms / (noise_rms + 1e-12)
    return float(20 * np.log10(snr_linear + 1e-12))

In [23]:
# ============================================================================
# CELL 3: STAGE 1 - PRE-PROCESSING AUDIT (Fixed np.trapezoid)
# ============================================================================

from scipy.signal import welch, find_peaks
from scipy.stats import median_abs_deviation
from scipy import stats
import warnings

class ERGAudit:
    """Pre-Processing Audit for ERG Signals - No filtering applied"""

    def __init__(self, config: ERGConfig = None):
        self.config = config or CONFIG

    def run_full_audit(self, signal_uv: np.ndarray, fs_hz: float,
                       electrode_type: str = 'contact_lens',
                       prestimulus_samples: int = 0,
                       age_group: str = None) -> Dict[str, Any]:

        signal_clean = np.nan_to_num(signal_uv)
        prestimulus_ms = prestimulus_samples * 1000.0 / fs_hz if prestimulus_samples > 0 else 0
        prestimulus_status, prestimulus_msg = self.config.get_prestimulus_status(prestimulus_ms)

        total_duration_ms = len(signal_uv) * 1000.0 / fs_hz
        iscev_compliance = validate_iscev_requirements(fs_hz, prestimulus_ms, total_duration_ms)
        snr_db = calculate_snr(signal_clean, fs_hz, prestimulus_samples)
        snr_status, snr_msg = self.config.get_electrode_snr_status(electrode_type, snr_db)

        bandwidth_result = self._analyze_bandwidth(signal_clean, fs_hz)
        op_result = self._analyze_oscillatory_potentials(signal_clean, fs_hz)
        mains_result = self._analyze_mains_interference(signal_clean, fs_hz)
        artifact_result = self._analyze_artifacts(signal_clean, fs_hz, prestimulus_samples)
        quality_grade, grade_score, grade_desc = self._assign_quality_grade(
            op_result, bandwidth_result, artifact_result, mains_result, snr_status)
        age_group_used = age_group if age_group is not None else self.config.DEFAULT_AGE_GROUP
        age_flags = self.config.get_age_group_flag(age_group_used)

        return {
            'prestimulus': {'status': prestimulus_status, 'message': prestimulus_msg, 'ms': prestimulus_ms},
            'iscev_compliance': iscev_compliance,
            'snr': {'db': round(snr_db, 1), 'electrode_type': electrode_type, 'status': snr_status, 'message': snr_msg},
            'bandwidth': bandwidth_result,
            'oscillatory_potentials': op_result,
            'mains_interference': mains_result,
            'artifacts': artifact_result,
            'quality': {'grade': quality_grade, 'score': grade_score, 'description': grade_desc},
            'age_handling': {'reference_source': age_flags[0], 'flag': age_flags[1]}
        }

    def _analyze_bandwidth(self, signal: np.ndarray, fs_hz: float) -> Dict[str, Any]:
        freqs, psd = welch(signal, fs=fs_hz, nperseg=min(1024, len(signal)))
        psd_db = 10 * np.log10(psd + 1e-30)

        noise_mask = (freqs >= 400) & (freqs <= 500)
        noise_floor_db = np.mean(psd_db[noise_mask]) if np.any(noise_mask) else np.mean(psd_db[freqs >= 300])

        threshold_db = noise_floor_db + 6
        high_mask = (freqs >= 100) & (freqs <= fs_hz/2)
        above_threshold = psd_db[high_mask] > threshold_db

        if np.any(above_threshold):
            freq_above = freqs[high_mask][above_threshold]
            estimated_high_hz = freq_above[-1] if len(freq_above) > 0 else 300.0
            if estimated_high_hz > 400:
                estimated_high_hz = 400.0
                confidence = "CAPPED"
            else:
                confidence = "HIGH" if estimated_high_hz >= 250 else "MEDIUM"
            hardware_cutoff_hz = estimated_high_hz
        else:
            estimated_high_hz = 300.0
            hardware_cutoff_hz = None
            confidence = "LOW"

        if confidence == "LOW":
            hardware_cutoff_hz = None

        return {'estimated_high_hz': round(estimated_high_hz, 0), 'hardware_cutoff_hz': hardware_cutoff_hz, 'confidence': confidence, 'noise_floor_db': round(noise_floor_db, 1)}

    def _analyze_oscillatory_potentials(self, signal: np.ndarray, fs_hz: float) -> Dict[str, Any]:
        freqs, psd = welch(signal, fs=fs_hz, nperseg=min(1024, len(signal)))

        op_mask = (freqs >= self.config.ISCEV_OP_BAND_LOW_HZ) & (freqs <= self.config.OP_BAND_HIGH_HZ)
        op_band_energy = np.trapezoid(psd[op_mask], freqs[op_mask])

        noise_mask = (freqs >= self.config.NOISE_BAND_LOW_HZ) & (freqs <= self.config.NOISE_BAND_HIGH_HZ)
        if np.any(noise_mask):
            noise_band_energy = np.trapezoid(psd[noise_mask], freqs[noise_mask])
            snr_db = 10 * np.log10((op_band_energy + 1e-30) / (noise_band_energy + 1e-30))
        else:
            noise_band_energy = 0
            snr_db = -np.inf

        if snr_db >= self.config.OP_BAND_SNR_THRESHOLD_DB:
            available = True
            quality = "excellent" if snr_db >= 10 else "good"
            reason = f"OP band SNR = {snr_db:.1f} dB (≥ threshold)"
        else:
            available = False
            quality = "absent"
            reason = f"OP band SNR = {snr_db:.1f} dB (< {self.config.OP_BAND_SNR_THRESHOLD_DB} dB threshold)"

        return {'available': available, 'quality': quality, 'snr_db': round(snr_db, 1), 'reason': reason}

    def _analyze_mains_interference(self, signal: np.ndarray, fs_hz: float) -> Dict[str, Any]:
        freqs, psd = welch(signal, fs=fs_hz, nperseg=min(1024, len(signal)))
        psd_db = 10 * np.log10(psd + 1e-30)

        noise_mask = np.ones_like(freqs, dtype=bool)
        for candidate in self.config.MAINS_CANDIDATE_FREQS_HZ:
            noise_mask &= ~((freqs >= candidate - 5) & (freqs <= candidate + 5))
        noise_floor_db = np.mean(psd_db[noise_mask]) if np.any(noise_mask) else -80

        results = {'detected_hz': None, 'excess_db': 0, 'confidence': 'NONE'}

        for candidate in self.config.MAINS_CANDIDATE_FREQS_HZ:
            idx = np.argmin(np.abs(freqs - candidate))
            peak_db = psd_db[idx]
            excess_db = peak_db - noise_floor_db

            if excess_db > self.config.MAINS_DETECTION_THRESHOLD_DB and excess_db > results['excess_db']:
                results['detected_hz'] = candidate
                results['excess_db'] = round(excess_db, 1)
                results['confidence'] = 'HIGH' if excess_db >= 10 else 'MEDIUM'

        results['recommended_action'] = 'SKIP (ISCEV 2022 compliant)' if results['detected_hz'] is None else f'DETECTED: {results["detected_hz"]} Hz - Notch filter NOT recommended'
        return results

    def _analyze_artifacts(self, signal: np.ndarray, fs_hz: float, prestimulus_samples: int = 0) -> Dict[str, Any]:
        mad = median_abs_deviation(signal)
        threshold = 5 * mad
        spikes = np.where(np.abs(signal - np.median(signal)) > threshold)[0]

        if len(spikes) > 0:
            spike_groups = []
            current_group = [spikes[0]]
            for i in range(1, len(spikes)):
                if spikes[i] - spikes[i-1] <= 5:
                    current_group.append(spikes[i])
                else:
                    spike_groups.append(current_group)
                    current_group = [spikes[i]]
            spike_groups.append(current_group)
            spike_count = len(spike_groups)
            max_spike_uv = float(np.max(np.abs(signal[spikes]))) if len(spikes) > 0 else 0
        else:
            spike_count = 0
            max_spike_uv = 0.0

        t = np.arange(len(signal)) / fs_hz * 1000
        slope, _, _, _, _ = stats.linregress(t, signal)
        drift_uv_per_ms = slope

        amplifier_ceiling_uv = 5000.0
        saturation = bool(
            np.any(signal >= amplifier_ceiling_uv * 0.98) or
            np.any(signal <= -amplifier_ceiling_uv * 0.98)
        )
        step_size = np.diff(signal)
        flat_run_fraction = np.sum(np.abs(step_size) < 0.01) / len(step_size) if len(step_size) > 0 else 0
        saturation = saturation or (flat_run_fraction > 0.05)

        # EMG detection per Chapter 6 §6.2.2 – High-frequency band (150-300 Hz)
        pre_n = max(prestimulus_samples, 8)
        pre_s = signal[:pre_n]

        if prestimulus_samples > 0:
            post_s = signal[prestimulus_samples:]
        else:
            post_s = signal[pre_n:]

        f_pre, p_pre = welch(pre_s, fs=fs_hz, nperseg=min(64, len(pre_s)))
        f_post, p_post = welch(post_s, fs=fs_hz, nperseg=min(64, len(post_s)))

        emg_mask_pre = (f_pre >= 150) & (f_pre <= 300)
        emg_mask_post = (f_post >= 150) & (f_post <= 300)

        emg_pre_energy = float(np.sum(p_pre[emg_mask_pre])) if emg_mask_pre.any() else 1e-12
        emg_post_energy = float(np.sum(p_post[emg_mask_post])) if emg_mask_post.any() else 1e-12

        excess_db = 10 * np.log10(emg_post_energy / (emg_pre_energy + 1e-30))

        if excess_db > 12:
            emg_level = "HIGH"
        elif excess_db > 8:
            emg_level = "MODERATE"
        else:
            emg_level = "LOW"

        emg_details = {
            "level": emg_level,
            "excess_db": round(excess_db, 1),
            "emg_pre_energy": round(emg_pre_energy, 4),
            "emg_post_energy": round(emg_post_energy, 4)
        }

        return {'spike_count': spike_count, 'max_spike_uv': round(max_spike_uv, 2), 'drift_uv_per_ms': round(drift_uv_per_ms, 4), 'saturation': saturation, 'emg_level': emg_level, 'emg_details': emg_details}

    def _assign_quality_grade(self, op_result, bandwidth_result, artifact_result, mains_result, snr_status):
        score = 0
        if op_result['available'] and op_result['quality'] == 'excellent':
            score += 3
        elif op_result['available']:
            score += 2
        if bandwidth_result['estimated_high_hz'] >= 250:
            score += 2
        elif bandwidth_result['estimated_high_hz'] >= 150:
            score += 1
        if artifact_result['spike_count'] == 0 and not artifact_result['saturation']:
            score += 2
        elif artifact_result['spike_count'] <= 5 and not artifact_result['saturation']:
            score += 1
        if snr_status == "PASS":
            score += 1
        if mains_result['detected_hz'] is None:
            score += 1

        if score >= 7:
            return "A", score, "Excellent - Full ISCEV bandwidth preserved, clean signal"
        elif score >= 5:
            return "B", score, "Good - Minor issues present, OP features likely usable"
        elif score >= 3:
            return "C", score, "Acceptable - OP features may be compromised"
        else:
            return "D", score, "Poor - Significant issues, OP features unavailable"


print("\n" + "=" * 60)
print("STAGE 1: PRE-PROCESSING AUDIT - READY")
print("=" * 60)
print("ERGAudit class instantiated. Ready to analyze signals.")
print("=" * 60)


STAGE 1: PRE-PROCESSING AUDIT - READY
ERGAudit class instantiated. Ready to analyze signals.


In [24]:
# ============================================================================
# CELL 4: STAGE 2 - CONDITIONAL FILTERING (ISCEV 2022 Compliant)
# ============================================================================

from scipy.signal import butter, sosfiltfilt, iirnotch, tf2sos, medfilt

class ERGFilter:
    """ISCEV 2022-Compliant ERG Filtering Pipeline"""

    def __init__(self, config: ERGConfig = None):
        self.config = config or CONFIG

    def design_bandpass(self, lowcut_hz: float, highcut_hz: float, fs_hz: float) -> np.ndarray:
        nyquist = fs_hz / 2.0
        if highcut_hz >= nyquist:
            raise ValueError(f"Low-pass cutoff {highcut_hz} Hz exceeds Nyquist ({nyquist} Hz)")
        low = lowcut_hz / nyquist
        high = highcut_hz / nyquist
        return butter(self.config.BUTTERWORTH_ORDER, [low, high], btype='bandpass', output='sos')

    def apply_bandpass(self, signal: np.ndarray, fs_hz: float, lowcut_hz: float = None, highcut_hz: float = None) -> np.ndarray:
        if lowcut_hz is None:
            lowcut_hz = self.config.ISCEV_HIGHPASS_HZ
        if highcut_hz is None:
            highcut_hz = self.config.ISCEV_LOWPASS_HZ
        sos = self.design_bandpass(lowcut_hz, highcut_hz, fs_hz)
        return sosfiltfilt(sos, signal)

    def apply_median(self, signal: np.ndarray, fs_hz: float) -> np.ndarray:
        kernel_samples = compute_kernel_samples(fs_hz, self.config.MEDIAN_KERNEL_MS)
        return medfilt(signal, kernel_size=kernel_samples)

    def design_notch(self, notch_hz: float, fs_hz: float) -> np.ndarray:
        b, a = iirnotch(notch_hz, self.config.NOTCH_QUALITY_FACTOR, fs=fs_hz)
        return tf2sos(b, a)

    def apply_notch(self, signal: np.ndarray, fs_hz: float, notch_hz: float) -> np.ndarray:
        sos = self.design_notch(notch_hz, fs_hz)
        return sosfiltfilt(sos, signal)

    def run_filter_pipeline(self, signal: np.ndarray, fs_hz: float,
                           apply_notch: bool = False, notch_hz: float = 50.0,
                           hardware_cutoff_hz: float = None,
                           user_confirmed_notch: bool = False) -> Tuple[np.ndarray, List[str]]:
        log = []
        sig = signal.copy()

        # Step 1: Median filter (always applied)
        sig_before = sig.copy()
        sig = self.apply_median(sig, fs_hz)
        spike_removed = np.sum(np.abs(sig_before - sig) > 5) > 0
        log.append(f"Median filter: kernel={self.config.MEDIAN_KERNEL_MS} ms, {compute_kernel_samples(fs_hz, self.config.MEDIAN_KERNEL_MS)} samples")
        if spike_removed:
            log.append("  - Spikes detected and removed")

        # Step 2: Notch filter (OFF by default - ISCEV 2022 compliance)
        if apply_notch and user_confirmed_notch:
            log.append(f"⚠️ NOTCH FILTER: User override at {notch_hz} Hz, Q={self.config.NOTCH_QUALITY_FACTOR}")
            sig = self.apply_notch(sig, fs_hz, notch_hz)
            log.append("   WARNING: Notch filters distort ERG waveform per ISCEV 2022")
        else:
            log.append(f"Notch filter: SKIPPED (ISCEV 2022 compliant - OFF by default)")

        # Step 3: Determine effective high-pass cutoff
        effective_high_hz = self.config.ISCEV_LOWPASS_HZ
        if hardware_cutoff_hz is not None and hardware_cutoff_hz < self.config.ISCEV_LOWPASS_HZ:
            effective_high_hz = hardware_cutoff_hz * 0.95
            log.append(f"Hardware cutoff detected: {hardware_cutoff_hz:.0f} Hz → effective: {effective_high_hz:.0f} Hz")

        # Step 4: Butterworth bandpass
        sig = self.apply_bandpass(sig, fs_hz, self.config.ISCEV_HIGHPASS_HZ, effective_high_hz)
        log.append(f"Butterworth bandpass: order={self.config.BUTTERWORTH_ORDER}, {self.config.ISCEV_HIGHPASS_HZ}-{effective_high_hz:.0f} Hz (zero-phase)")

        return sig, log

    def extract_ops(self, signal: np.ndarray, fs_hz: float) -> np.ndarray:
        sos = self.design_bandpass(self.config.ISCEV_OP_BAND_LOW_HZ, self.config.OP_BAND_HIGH_HZ, fs_hz)
        return sosfiltfilt(sos, signal)


print("\n" + "=" * 60)
print("STAGE 2: CONDITIONAL FILTERING - READY")
print("=" * 60)
print("Filter pipeline: Median → Notch (OFF by default) → Butterworth Bandpass")
print("ISCEV 2022: Notch filter OFF by default (Page 7, Col 2, Para 4)")
print("=" * 60)


STAGE 2: CONDITIONAL FILTERING - READY
Filter pipeline: Median → Notch (OFF by default) → Butterworth Bandpass
ISCEV 2022: Notch filter OFF by default (Page 7, Col 2, Para 4)


In [25]:
# ============================================================================
# CELL 5: STAGE 3 - FEATURE EXTRACTION (with flash midpoint correction)
# Updated: Protocol-specific windows, OP4 window 35-65 ms, OP sum = OP2+OP3+OP4, PhNR added
# FIXED: a-wave forced negative, b-wave forced positive, b-wave after a-wave enforced
# ============================================================================

from scipy.signal import find_peaks

class ERGFeatureExtractor:
    """ISCEV 2022-Compliant Feature Extraction"""

    def __init__(self, config: ERGConfig = None):
        self.config = config or CONFIG

    def extract_awave_bwave(self, signal: np.ndarray, fs_hz: float,
                            protocol: str = 'DA 3',
                            flash_onset_sample: int = 0,
                            flash_duration_ms: float = 1.0) -> Dict[str, Any]:

        # Protocol-specific search windows per Action Plan §3.4.1
        if protocol == 'DA 0.01':
            a_wave_window_ms = (None, None)
            b_wave_window_ms = (40, 130)
        elif protocol == 'DA 3':
            a_wave_window_ms = (10, 35)
            b_wave_window_ms = (30, 100)
        elif protocol == 'DA 10':
            a_wave_window_ms = (8, 30)
            b_wave_window_ms = (25, 95)
        elif protocol == 'LA 3':
            a_wave_window_ms = (12, 28)
            b_wave_window_ms = (25, 60)
        elif protocol == 'LA 30 Hz':
            a_wave_window_ms = (None, None)
            b_wave_window_ms = (20, 60)
        else:
            a_wave_window_ms = (10, 35)
            b_wave_window_ms = (30, 100)

        # Handle protocols with no a-wave (DA 0.01, LA 30 Hz)
        if a_wave_window_ms[0] is None:
            a_wave_amplitude = np.nan
            a_wave_implicit_ms = np.nan
            # Still need b-wave start/end
            b_start = flash_onset_sample + int(b_wave_window_ms[0] * fs_hz / 1000)
            b_end = flash_onset_sample + int(b_wave_window_ms[1] * fs_hz / 1000)
            b_start = max(0, min(b_start, len(signal)-1))
            b_end = max(b_start+1, min(b_end, len(signal)))
        else:
            a_start = flash_onset_sample + int(a_wave_window_ms[0] * fs_hz / 1000)
            a_end = flash_onset_sample + int(a_wave_window_ms[1] * fs_hz / 1000)

            a_start = max(0, min(a_start, len(signal)-1))
            a_end = max(a_start+1, min(a_end, len(signal)))

            a_segment = signal[a_start:a_end]
            if len(a_segment) > 0:
                a_trough_idx_local = np.argmin(a_segment)
                a_trough_idx = a_start + a_trough_idx_local
                a_wave_amplitude = signal[a_trough_idx]
                # FORCE a-wave to be negative (physiological constraint)
                if a_wave_amplitude > 0:
                    a_wave_amplitude = -a_wave_amplitude
                a_wave_implicit_samples = a_trough_idx - flash_onset_sample
                a_wave_implicit_ms = a_wave_implicit_samples * 1000.0 / fs_hz
            else:
                a_wave_amplitude = np.nan
                a_wave_implicit_ms = np.nan

                # CRITICAL FIX: b-wave search must start AFTER a-wave trough
            if not np.isnan(a_wave_implicit_ms):
                b_start_abs = max(a_trough_idx + int(2 * fs_hz / 1000),
                                  flash_onset_sample + int(b_wave_window_ms[0] * fs_hz / 1000))
            else:
                b_start_abs = flash_onset_sample + int(b_wave_window_ms[0] * fs_hz / 1000)

            b_end_abs = flash_onset_sample + int(b_wave_window_ms[1] * fs_hz / 1000)

            b_start = max(0, min(b_start_abs, len(signal)-1))
            b_end = max(b_start+1, min(b_end_abs, len(signal)))

        b_segment = signal[b_start:b_end]
        if len(b_segment) > 0:
            b_peak_idx_local = np.argmax(b_segment)
            b_peak_idx = b_start + b_peak_idx_local
            b_wave_raw = signal[b_peak_idx]
            b_wave_amplitude = b_wave_raw - a_wave_amplitude if not np.isnan(a_wave_amplitude) else b_wave_raw
            # Ensure b-wave amplitude is positive
            if b_wave_amplitude < 0:
                b_wave_amplitude = -b_wave_amplitude
            b_wave_implicit_samples = b_peak_idx - flash_onset_sample
            b_wave_implicit_ms = b_wave_implicit_samples * 1000.0 / fs_hz
        else:
            b_wave_amplitude = np.nan
            b_wave_implicit_ms = np.nan

        # Ensure a-wave amplitude is reported as negative (clinical convention)
        if not np.isnan(a_wave_amplitude) and a_wave_amplitude > 0:
            a_wave_amplitude = -a_wave_amplitude

        ba_ratio = b_wave_amplitude / abs(a_wave_amplitude) if a_wave_amplitude != 0 and not np.isnan(a_wave_amplitude) else np.nan

        correction_applied = False
        correction_ms = 0.0

        if flash_duration_ms >= self.config.ISCEV_FLASH_MAX_DURATION_MS:
            correction_ms = flash_duration_ms / 2.0
            a_wave_implicit_ms = a_wave_implicit_ms - correction_ms if not np.isnan(a_wave_implicit_ms) else np.nan
            b_wave_implicit_ms = b_wave_implicit_ms - correction_ms if not np.isnan(b_wave_implicit_ms) else np.nan
            correction_applied = True

        return {
            'a_wave_amplitude_uv': round(float(a_wave_amplitude), 1) if not np.isnan(a_wave_amplitude) else np.nan,
            'a_wave_implicit_time_ms': round(float(a_wave_implicit_ms), 1) if not np.isnan(a_wave_implicit_ms) else np.nan,
            'b_wave_amplitude_uv': round(float(b_wave_amplitude), 1) if not np.isnan(b_wave_amplitude) else np.nan,
            'b_wave_implicit_time_ms': round(float(b_wave_implicit_ms), 1) if not np.isnan(b_wave_implicit_ms) else np.nan,
            'ba_ratio': round(float(ba_ratio), 2) if not np.isnan(ba_ratio) else np.nan,
            'flash_midpoint_correction_applied': correction_applied,
            'flash_midpoint_correction_ms': correction_ms
        }

    def extract_oscillatory_potentials(self, signal: np.ndarray, fs_hz: float,
                                        flash_onset_sample: int = 0) -> Dict[str, Any]:
        """
        Extract Oscillatory Potentials OP1-OP4 from 75-300 Hz filtered signal
        Per ISCEV 2022: OP4 window 35-65 ms, OP sum = OP2+OP3+OP4 (excludes OP1)
        """

        # OP search windows (post-flash in ms) per ISCEV 2022
        # OP4 window extended to 35-65 ms (peaks at 50-65 ms in human dark-adapted ERG)
        op_windows = {
            'OP1': (12, 20),
            'OP2': (20, 28),
            'OP3': (28, 38),
            'OP4': (38, 65)  # Non-overlapping with OP3
        }

        ops = {}

        for op_name, (start_ms, end_ms) in op_windows.items():
            start_idx = flash_onset_sample + int(start_ms * fs_hz / 1000)
            end_idx = flash_onset_sample + int(end_ms * fs_hz / 1000)

            start_idx = max(0, min(start_idx, len(signal)-1))
            end_idx = max(start_idx+1, min(end_idx, len(signal)))

            segment = signal[start_idx:end_idx]

            if len(segment) > 0:
                peaks, _ = find_peaks(segment, height=0, prominence=0.5)
                if len(peaks) > 0:
                    peak_idx_local = peaks[np.argmax(segment[peaks])]
                    peak_amplitude = segment[peak_idx_local]
                    trough_idx_local = np.argmin(segment[:peak_idx_local+1]) if peak_idx_local > 0 else 0
                    trough_amplitude = segment[trough_idx_local]
                    op_amplitude = peak_amplitude - trough_amplitude
                else:
                    op_amplitude = 0.0
            else:
                op_amplitude = 0.0

            ops[op_name] = round(float(op_amplitude), 1)

        # OP sum per ISCEV: OP2+OP3+OP4 (excludes OP1)
        ops['OP_sum_uv'] = ops.get('OP2', 0) + ops.get('OP3', 0) + ops.get('OP4', 0)

        return ops

    def extract_phnr(self, signal: np.ndarray, fs_hz: float,
                     b_wave_implicit_time_ms: float,
                     flash_onset_sample: int = 0,
                     noise_rms_uv: float = 1.0) -> Dict[str, Any]:
        """
        Extract PhNR amplitude from LA 3.0 broadband-filtered signal
        Per Chapter 9 §9.1.4: search 60-200 ms post-stimulus window
        """
        # Search window after b-wave peak (60-200 ms post-stimulus)
        search_start_ms = max(60.0, b_wave_implicit_time_ms + 10.0)
        search_end_ms = 200.0

        start_idx = flash_onset_sample + int(search_start_ms * fs_hz / 1000)
        end_idx = flash_onset_sample + int(search_end_ms * fs_hz / 1000)

        start_idx = max(0, min(start_idx, len(signal)-1))
        end_idx = max(start_idx+1, min(end_idx, len(signal)))

        segment = signal[start_idx:end_idx]

        if len(segment) > 0:
            phnr_trough = float(np.min(segment))
            if phnr_trough >= 0:
                return {'phnr_amp_uv': np.nan}
            phnr_amp = abs(phnr_trough)
            # Reliability check: PhNR must exceed 2x noise RMS
            if phnr_amp < 2.0 * noise_rms_uv:
                return {'phnr_amp_uv': np.nan}
            return {'phnr_amp_uv': round(phnr_amp, 1)}
        else:
            return {'phnr_amp_uv': np.nan}

    def extract_all_features(self, signal: np.ndarray, fs_hz: float,
                             protocol: str = 'DA 3',
                             flash_onset_sample: int = 0,
                             flash_duration_ms: float = 1.0,
                             op_signal: np.ndarray = None,
                             noise_rms_uv: float = 1.0) -> Dict[str, Any]:

        features = {'protocol': protocol}

        if '30' not in protocol:
            awave_bwave = self.extract_awave_bwave(signal, fs_hz, protocol, flash_onset_sample, flash_duration_ms)
            features.update(awave_bwave)

        if op_signal is not None:
            ops = self.extract_oscillatory_potentials(op_signal, fs_hz, flash_onset_sample)
            features['oscillatory_potentials'] = ops

        # Extract PhNR for LA 3.0 protocol
        if protocol == 'LA 3':
            if 'b_wave_implicit_time_ms' in features and not np.isnan(features['b_wave_implicit_time_ms']):
                phnr = self.extract_phnr(signal, fs_hz, features['b_wave_implicit_time_ms'],
                                         flash_onset_sample, noise_rms_uv)
                features.update(phnr)
            else:
                features['phnr_amp_uv'] = np.nan

        return features


print("\n" + "=" * 60)
print("STAGE 3: FEATURE EXTRACTION - READY")
print("=" * 60)
print("Features: a-wave amplitude, a-wave implicit time, b-wave amplitude,")
print("          b-wave implicit time, b/a ratio, flash midpoint correction")
print("          Oscillatory Potentials (OP1-OP4) with OP sum = OP2+OP3+OP4")
print("          PhNR amplitude (LA 3.0 protocol)")
print("=" * 60)


STAGE 3: FEATURE EXTRACTION - READY
Features: a-wave amplitude, a-wave implicit time, b-wave amplitude,
          b-wave implicit time, b/a ratio, flash midpoint correction
          Oscillatory Potentials (OP1-OP4) with OP sum = OP2+OP3+OP4
          PhNR amplitude (LA 3.0 protocol)


In [26]:
# ============================================================================
# CELL 6: STAGE 4 - COMPLIANCE REPORT WITH BAKER ET AL. 2025 REFERENCE RANGES
# Fully updated: Age-stratified reference ranges from 407 healthy subjects
# Reference: Baker RA et al. Doc Ophthalmol (2025) 150:47–64
# DOI: 10.1007/s10633-025-10009-2
# Version: 2.3.2
# Date: 18 May 2026
# ============================================================================

from datetime import datetime
from typing import Dict, Any, List, Tuple, Optional
import numpy as np

class ERGReportGenerator:
    """Generates ISCEV 2022 Compliance Report with Baker et al. 2025 reference ranges
       Silver thread electrodes (fornix position) — age-stratified (≤35, 36-59, ≥60)
       Gold foil data already transformed to silver thread equivalents
    """

    def __init__(self, config: ERGConfig = None):
        self.config = config or CONFIG

        # ====================================================================
        # REFERENCE RANGES FROM BAKER ET AL. 2025 — AGE-STRATIFIED
        # Silver thread electrodes (fornix position) — parametric means ± SD
        # Derived from 407 healthy subjects (age 7-86, 83.5% female)
        # Amplitude values are POSITIVE (ISCEV convention)
        # CORRECTED per peer review (15 values updated 18 May 2026)
        # ====================================================================

        self.REFERENCE_RANGES = {
            # ==================== DA 0.01 (Rod response) ====================
            'DA 0.01': {
                '≤35': {'a_amp': (0, 0), 'a_imp': (0, 0),
                        'b_amp': (191, 38), 'b_imp': (85, 8)},
                '36-59': {'a_amp': (0, 0), 'a_imp': (0, 0),
                          'b_amp': (190, 48), 'b_imp': (92, 9)},
                '≥60': {'a_amp': (0, 0), 'a_imp': (0, 0),
                        'b_amp': (190, 50), 'b_imp': (96, 10)},
            },

            # ==================== DA 3 (Maximal combined) ====================
            'DA 3': {
                '≤35': {'a_amp': (175, 38), 'a_imp': (15.0, 0.6),
                        'b_amp': (266, 50), 'b_imp': (51, 4)},
                '36-59': {'a_amp': (155, 38), 'a_imp': (15.5, 1.0),
                          'b_amp': (280, 55), 'b_imp': (50.6, 2.4)},
                '≥60': {'a_amp': (140, 40), 'a_imp': (16.0, 0.8),
                        'b_amp': (280, 60), 'b_imp': (52, 4)},
            },

            # ==================== DA 10 (Strong flash) ====================
            'DA 10': {
                '≤35': {'a_amp': (200, 42), 'a_imp': (11.8, 0.9),
                        'b_amp': (276, 49), 'b_imp': (50, 4)},
                '36-59': {'a_amp': (190, 45), 'a_imp': (12.5, 1.0),
                          'b_amp': (290, 55), 'b_imp': (51, 4)},
                '≥60': {'a_amp': (170, 45), 'a_imp': (13.3, 1.0),
                        'b_amp': (290, 60), 'b_imp': (51, 4)},
            },

            # ==================== LA 3 (Single flash cone) ====================
            'LA 3': {
                '≤35': {'a_amp': (28, 8), 'a_imp': (14.0, 1.0),
                        'b_amp': (120, 40), 'b_imp': (29, 2)},
                '36-59': {'a_amp': (25, 8), 'a_imp': (14.5, 1.0),
                          'b_amp': (110, 35), 'b_imp': (29, 2)},
                '≥60': {'a_amp': (22, 7), 'a_imp': (14.2, 0.6),
                        'b_amp': (95, 35), 'b_imp': (30, 2)},
            },

            # ==================== LA 30 Hz (Flicker cone) ====================
            'LA 30 Hz': {
                '≤35': {'a_amp': (0, 0), 'a_imp': (0, 0),
                        'b_amp': (110, 30), 'b_imp': (25.4, 0.9)},
                '36-59': {'a_amp': (0, 0), 'a_imp': (0, 0),
                          'b_amp': (85, 25), 'b_imp': (25.7, 1.2)},
                '≥60': {'a_amp': (0, 0), 'a_imp': (0, 0),
                        'b_amp': (75, 25), 'b_imp': (26.4, 1.4)},
            },
        }

        # ====================================================================
        # b/a RATIO REFERENCE RANGES (NOT in Baker 2025 — retained estimates)
        # ====================================================================

        self.BA_RATIO_REFERENCE = {
            'contact_lens': (4.0, 0.8),
            'gold_foil': (4.0, 0.8),
            'dtl_fiber': (4.5, 1.0),
            'skin': (6.0, 1.2),
        }

        # ====================================================================
        # PhNR REFERENCE RANGES (NOT in Baker 2025 — retained estimates)
        # ====================================================================

        self.PHNR_REFERENCE_RANGES = {
            'contact_lens': {'phnr_amp': (12, 4)},
            'gold_foil':    {'phnr_amp': (10, 3.5)},
            'dtl_fiber':    {'phnr_amp': (8, 3)},
            'skin':         {'phnr_amp': (4, 2)},
        }

        # ====================================================================
        # ELECTRODE MAPPING
        # ====================================================================

        self.ELECTRODE_MAPPING = {
            'Silver thread (fornix)': 'contact_lens',
            'contact_lens': 'contact_lens',
            'DTL': 'contact_lens',
            'dtl_fiber': 'contact_lens',
            'Gold foil - transformed': 'contact_lens',  # Already transformed
            'gold_foil': 'contact_lens',
            'Gold foil': 'contact_lens',
            'Skin': 'skin',
            'skin': 'skin',
        }

    def _normalize_electrode_type(self, electrode_type: Any) -> str:
        """Convert various electrode input formats to internal key"""
        if isinstance(electrode_type, dict):
            electrode_type = electrode_type.get('value', 'contact_lens')
        elif not isinstance(electrode_type, str):
            electrode_type = str(electrode_type)
        return self.ELECTRODE_MAPPING.get(electrode_type, 'contact_lens')

    def _get_age_group(self, age: Optional[float]) -> str:
        """Determine age group per Baker 2025 stratification"""
        if age is None:
            return '36-59'  # Default to middle group
        if age <= 35:
            return '≤35'
        elif age <= 59:
            return '36-59'
        else:
            return '≥60'

    def calculate_z_score(self, value: float, mean: float, sd: float) -> float:
        """Calculate Z-score following ISCEV convention"""
        if value is None or np.isnan(value):
            return 0.0
        if sd == 0 or mean == 0:
            return 0.0
        return round((value - mean) / sd, 2)

    def compute_z_scores(self, features: Dict, electrode_type: str,
                         protocol: str, age: Optional[float] = None) -> Dict[str, float]:
        """Compute Z-scores using Baker 2025 age-stratified reference ranges"""

        electrode_type = self._normalize_electrode_type(electrode_type)

        # Get protocol and age-appropriate reference ranges
        proto_ranges = self.REFERENCE_RANGES.get(protocol, self.REFERENCE_RANGES['DA 3'])
        age_group = self._get_age_group(age)
        ref = proto_ranges.get(age_group, proto_ranges['36-59'])

        # For LA 30 Hz, b_amp/b_imp are used as peak amplitude/timing
        z_scores = {}

        # a-wave amplitude
        if features.get('a_wave_amplitude_uv') and ref['a_amp'][0] > 0:
            z_scores['a_wave_amplitude'] = self.calculate_z_score(
                abs(features['a_wave_amplitude_uv']), ref['a_amp'][0], ref['a_amp'][1])

        # a-wave implicit time
        if features.get('a_wave_implicit_time_ms') and ref['a_imp'][0] > 0:
            z_scores['a_wave_implicit_time'] = self.calculate_z_score(
                features['a_wave_implicit_time_ms'], ref['a_imp'][0], ref['a_imp'][1])

        # b-wave amplitude (or peak amplitude for LA 30 Hz)
        if features.get('b_wave_amplitude_uv') or features.get('peak_amplitude_uv'):
            amp = features.get('b_wave_amplitude_uv') or features.get('peak_amplitude_uv')
            z_scores['b_wave_amplitude'] = self.calculate_z_score(amp, ref['b_amp'][0], ref['b_amp'][1])

        # b-wave implicit time (or peak timing for LA 30 Hz)
        if features.get('b_wave_implicit_time_ms') or features.get('peak_implicit_time_ms'):
            imp = features.get('b_wave_implicit_time_ms') or features.get('peak_implicit_time_ms')
            z_scores['b_wave_implicit_time'] = self.calculate_z_score(imp, ref['b_imp'][0], ref['b_imp'][1])

        # b/a ratio (non-Baker estimate)
        if features.get('ba_ratio'):
            ref_mean, ref_sd = self.BA_RATIO_REFERENCE.get(electrode_type, (4.0, 0.8))
            z_scores['ba_ratio'] = self.calculate_z_score(features['ba_ratio'], ref_mean, ref_sd)

        # PhNR (non-Baker estimate)
        if features.get('phnr_amp_uv') and protocol == 'LA 3':
            phnr_ref = self.PHNR_REFERENCE_RANGES.get(electrode_type, {'phnr_amp': (18, 6)})['phnr_amp']
            z_scores['phnr_amplitude'] = self.calculate_z_score(
                features['phnr_amp_uv'], phnr_ref[0], phnr_ref[1])

        return z_scores

    def generate_traffic_light(self, z_scores: Dict[str, float]) -> Dict[str, Any]:
        """Generate traffic light: GREEN ≤2, AMBER 2-3, RED >3"""

        if not z_scores:
            return {
                'signal': 'YELLOW', 'color': '🟡',
                'message': 'Insufficient data for Z-score calculation',
                'confidence': 0.5, 'max_z_score': 0,
                'worst_parameter': None, 'worst_z_score': 0
            }

        max_abs_z = 0
        worst_param = None
        worst_value = 0

        for param, z in z_scores.items():
            if z is not None and abs(z) > max_abs_z:
                max_abs_z = abs(z)
                worst_param = param
                worst_value = z

        if max_abs_z <= 2.0:
            return {
                'signal': 'GREEN', 'color': '🟢',
                'message': f'All ERG parameters within normal limits (|Z|={max_abs_z:.2f} ≤ 2.0). No immediate action required.',
                'confidence': 0.95, 'max_z_score': max_abs_z,
                'worst_parameter': worst_param, 'worst_z_score': worst_value
            }
        elif max_abs_z <= 3.0:
            direction = "delayed" if worst_param and 'implicit' in worst_param and worst_value > 0 else \
                       "shortened" if worst_param and 'implicit' in worst_param else \
                       "reduced" if worst_value < 0 else "elevated"
            return {
                'signal': 'AMBER', 'color': '🟡',
                'message': f'Borderline abnormality: {worst_param.replace("_", " ")} {direction} (Z={max_abs_z:.2f}). Specialist review recommended.',
                'confidence': 0.85, 'max_z_score': max_abs_z,
                'worst_parameter': worst_param, 'worst_z_score': worst_value
            }
        else:
            direction = "delayed" if worst_param and 'implicit' in worst_param and worst_value > 0 else \
                       "shortened" if worst_param and 'implicit' in worst_param else \
                       "reduced" if worst_value < 0 else "elevated"
            return {
                'signal': 'RED', 'color': '🔴',
                'message': f'Significant abnormality: {worst_param.replace("_", " ")} {direction} (Z={max_abs_z:.2f}). Urgent review required.',
                'confidence': 0.90, 'max_z_score': max_abs_z,
                'worst_parameter': worst_param, 'worst_z_score': worst_value
            }

    def generate_full_report(self, features: Dict, audit_results: Dict,
                             electrode_type: str, protocol: str,
                             filtered_signal: np.ndarray,
                             time_ms: np.ndarray,
                             filter_log: List[str],
                             processing_time_ms: float,
                             patient_age: Optional[float] = None) -> Dict[str, Any]:
        """Generate complete four-layer ERG clinical report"""

        electrode_type_norm = self._normalize_electrode_type(electrode_type)
        age_group = self._get_age_group(patient_age)

        # Compute Z-scores with age
        z_scores = self.compute_z_scores(features, electrode_type, protocol, patient_age)
        traffic_light = self.generate_traffic_light(z_scores)

        report_id = f"ERG-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

        # Key findings
        key_findings = []
        for param, z in z_scores.items():
            if abs(z) > 1.5:
                param_name = param.replace('_', ' ').title()
                if z > 0:
                    key_findings.append(f"{param_name} increased/delayed (Z={z:+.2f})")
                else:
                    key_findings.append(f"{param_name} reduced (Z={z:+.2f})")

        return {
            'layer_1_traffic_light': traffic_light,
            'layer_2_clinical_summary': {
                'traffic_signal': traffic_light['signal'],
                'quality_grade': audit_results.get('quality', {}).get('grade', 'unknown'),
                'key_findings': key_findings,
                'z_scores': z_scores,
                'recommended_action': traffic_light['message'],
                'disclaimer': "ERG interpretation using Baker et al. 2025 reference ranges (Silver thread, fornix). Age-stratified norms (≤35, 36-59, ≥60). Must interpret in full clinical context."
            },
            'layer_3_specialist': {
                'status': 'PENDING - Full implementation requires Chapter 8 (STFT) and Chapter 14 (SHAP)',
                'stft_spectrogram': None,
                'annotated_waveform': None,
                'shap_waterfall': None,
                'normative_comparison': {
                    'reference_source': 'Baker et al. 2025 (Doc Ophthalmol 150:47-64)',
                    'doi': '10.1007/s10633-025-10009-2',
                    'electrode_type': electrode_type_norm,
                    'protocol': protocol,
                    'age_group': age_group,
                    'n_subjects': 407,
                    'age_range': '7-86 years',
                    'sex_distribution': '83.5% female'
                }
            },
            'layer_4_technical_audit': {
                'report_id': report_id,
                'timestamp': datetime.now().isoformat(),
                'pipeline_version': '2.3.2',
                'electrode_type_input': electrode_type,
                'electrode_type_normalized': electrode_type_norm,
                'protocol': protocol,
                'patient_age': patient_age,
                'age_group_used': age_group,
                'z_scores': z_scores,
                'filter_log': filter_log,
                'processing_time_ms': processing_time_ms
            }
        }


# ============================================================================
# INITIALIZATION VERIFICATION
# ============================================================================

print("\n" + "=" * 70)
print("STAGE 4: COMPLIANCE REPORT - BAKER ET AL. 2025 COMPLIANT")
print("=" * 70)
print("Version: 2.3.2 | Date: 18 May 2026")
print("Reference source: Baker RA et al. Doc Ophthalmol (2025) 150:47-64")
print("DOI: 10.1007/s10633-025-10009-2")
print("-" * 70)
print("Reference ranges: Age-stratified (≤35, 36-59, ≥60)")
print("Electrodes: Silver thread (fornix) + Gold foil (transformed)")
print("Sample: n=407 healthy adults (age 7-86, 83.5% female)")
print("-" * 70)
print("⚠️  ba_ratio and PhNR: Retained estimates (not in Baker 2025)")
print("-" * 70)
print("Four-layer report: Layer 1 (Traffic), Layer 2 (Clinical),")
print("                  Layer 3 (Specialist - PENDING), Layer 4 (Audit)")
print("=" * 70)

# ============================================================================
# END OF UPDATED CELL 6 - V2.3.2
# ============================================================================


STAGE 4: COMPLIANCE REPORT - BAKER ET AL. 2025 COMPLIANT
Version: 2.3.2 | Date: 18 May 2026
Reference source: Baker RA et al. Doc Ophthalmol (2025) 150:47-64
DOI: 10.1007/s10633-025-10009-2
----------------------------------------------------------------------
Reference ranges: Age-stratified (≤35, 36-59, ≥60)
Electrodes: Silver thread (fornix) + Gold foil (transformed)
Sample: n=407 healthy adults (age 7-86, 83.5% female)
----------------------------------------------------------------------
⚠️  ba_ratio and PhNR: Retained estimates (not in Baker 2025)
----------------------------------------------------------------------
Four-layer report: Layer 1 (Traffic), Layer 2 (Clinical),
                  Layer 3 (Specialist - PENDING), Layer 4 (Audit)


In [27]:
# ============================================================================
# CELL 7: STFT SPECTROGRAM GENERATION (Chapter 8)
# Generates normalized 2D spectrograms for deep learning classifiers
# Default parameters: Hamming window, 64 samples, 56 overlap, 224x224 output
# ============================================================================

import numpy as np
from scipy.signal import stft
from skimage.transform import resize

# ============================================================================
# STFT Configuration (matches Chapter 8 §8.4.1)
# All parameters stored for config-API reproducibility
# ============================================================================

STFT_CONFIG = {
    'window': 'hamming',      # Default for deep learning (-43 dB sidelobe)
    'nperseg': 64,            # 32 ms window at 2000 Hz
    'noverlap': 56,           # 87.5% overlap, 4 ms hop
    'fs': 2000.0,             # Hz; must match recording metadata
    'fmin_hz': 0.0,           # Full ISCEV passband
    'fmax_hz': 300.0,         # ISCEV upper cutoff
    'output_h': 224,          # ViT input height
    'output_w': 224,          # ViT input width
    'norm': 'db_zscore',      # Z-score normalization (amplitude-invariant)
}


def generate_erg_spectrogram(signal_uv: np.ndarray,
                             config: dict = None,
                             return_axes: bool = False) -> np.ndarray:
    """
    Generate a normalized 2D spectrogram array from a filtered ERG sweep.

    Parameters
    ----------
    signal_uv : 1-D filtered ERG array in µV (output of Chapter 5 pipeline)
    config : STFT configuration dict. Defaults to STFT_CONFIG above.
    return_axes : If True, also return (freqs, times) for axis labelling.

    Returns
    -------
    S_norm : 2-D float32 array, shape (output_h, output_w), values in [0, 1]
             Ready for use as single-channel image input for ViT/CNN.

    Chapter 8 Reference: §8.4.1 Spectrogram Generation
    """
    cfg = config or STFT_CONFIG

    fs = cfg['fs']

    # Step 1: Compute STFT (Chapter 8, Equation 8.1)
    freqs, times, Zxx = stft(
        signal_uv,
        fs=fs,
        window=cfg['window'],
        nperseg=cfg['nperseg'],
        noverlap=cfg['noverlap']
    )

    # Step 2: Convert to power in dB
    # |STFT|^2 is the power spectrogram
    power = np.abs(Zxx) ** 2
    S_db = 10 * np.log10(power + 1e-12)  # epsilon avoids log(0)

    # Step 3: Crop to clinical frequency range (0-300 Hz per ISCEV)
    fmask = (freqs >= cfg['fmin_hz']) & (freqs <= cfg['fmax_hz'])
    S_db = S_db[fmask, :]
    freqs_cropped = freqs[fmask]

    # Step 4: Normalization (db_zscore per Chapter 8 §8.4.1)
    if cfg['norm'] == 'db_zscore':
        mu = S_db.mean()
        sigma = S_db.std() + 1e-8
        S_n = (S_db - mu) / sigma
        # Clip to ±3 sigma and rescale to [0, 1]
        S_n = np.clip(S_n, -3, 3)
        S_n = (S_n + 3) / 6.0
    elif cfg['norm'] == 'minmax':
        lo, hi = S_db.min(), S_db.max()
        S_n = (S_db - lo) / (hi - lo + 1e-8)
    else:
        raise ValueError(f"Unknown normalization: {cfg['norm']}")

    # Step 5: Resize to fixed output dimensions (bicubic interpolation)
    S_norm = resize(
        S_n,
        (cfg['output_h'], cfg['output_w']),
        order=3,               # bicubic
        anti_aliasing=True,
        preserve_range=True
    ).astype(np.float32)

    if return_axes:
        # Return time axis for plotting (ms, aligned to stimulus)
        times_ms = times * 1000
        return S_norm, freqs_cropped, times_ms

    return S_norm


def flatten_spectrogram_for_ml(S_norm: np.ndarray, small_size: int = 32) -> np.ndarray:
    """
    Downsample a normalized 224x224 spectrogram to small_size x small_size
    and flatten to a 1-D feature vector for classical ML input.

    Chapter 8 Reference: §8.4.2 Spectrogram Flattening for Classical ML
    """
    S_small = resize(
        S_norm,
        (small_size, small_size),
        order=3,
        anti_aliasing=True,
        preserve_range=True
    )
    return S_small.ravel().astype(np.float32)


class SpectrogramPCAReducer:
    """
    PCA-based dimensionality reducer for spectrogram feature vectors.
    Fit on training set; apply to test and deployment recordings.

    Chapter 8 Reference: §8.4.2
    """

    def __init__(self, n_components: int = 128, small_size: int = 32):
        from sklearn.decomposition import PCA
        from sklearn.preprocessing import StandardScaler

        self.pca = PCA(n_components=n_components, random_state=42)
        self.scaler = StandardScaler()
        self.small_size = small_size

    def fit(self, spectrograms: list):
        """Fit PCA on training spectrograms"""
        X = np.array([flatten_spectrogram_for_ml(S, self.small_size) for S in spectrograms])
        X_scaled = self.scaler.fit_transform(X)
        self.pca.fit(X_scaled)

        explained = self.pca.explained_variance_ratio_.cumsum()
        print(f"PCA: {self.pca.n_components_} components explain {100*explained[-1]:.1f}% of variance")
        return self

    def transform(self, spectrograms: list) -> np.ndarray:
        """Transform spectrograms using fitted PCA"""
        X = np.array([flatten_spectrogram_for_ml(S, self.small_size) for S in spectrograms])
        X_scaled = self.scaler.transform(X)
        return self.pca.transform(X_scaled)

    def fit_transform(self, spectrograms: list) -> np.ndarray:
        return self.fit(spectrograms).transform(spectrograms)


def spectrogram_to_tensor(S_norm: np.ndarray, channels: int = 3, imagenet_norm: bool = True):
    """
    Convert a normalized spectrogram array to a PyTorch image tensor.

    Parameters
    ----------
    S_norm : (H, W) float32 array, values in [0, 1]
    channels : 3 for pre-trained ViT/CNN (channel replication)
               1 for scratch-trained single-channel models
    imagenet_norm : If True, apply ImageNet mean/std normalization

    Chapter 8 Reference: §8.4.3 Spectrogram as Image Input for Vision Transformers
    """
    import torch

    if channels == 3:
        t = torch.from_numpy(S_norm).unsqueeze(0).repeat(3, 1, 1)  # (3, H, W)
    elif channels == 1:
        t = torch.from_numpy(S_norm).unsqueeze(0)  # (1, H, W)
    else:
        raise ValueError(f"channels must be 1 or 3, got {channels}")

    # ImageNet normalization for pre-trained models
    if imagenet_norm and channels == 3:
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        t = (t - mean) / std

    return t  # shape (C, H, W)


def batch_to_tensor(spectrograms: list, channels: int = 3, imagenet_norm: bool = True):
    """Convert a list of spectrograms to a (N, C, H, W) batch tensor"""
    import torch
    tensors = [spectrogram_to_tensor(S, channels, imagenet_norm) for S in spectrograms]
    return torch.stack(tensors)  # (N, C, H, W)


def augment_spectrogram(S_norm: np.ndarray,
                        time_mask_max: int = 30,
                        freq_mask_max: int = 20,
                        noise_std_max: float = 0.05,
                        rng: np.random.Generator = None) -> np.ndarray:
    """
    Apply SpecAugment-inspired augmentation to an ERG spectrogram.
    All augmentations are valid for ERG data (see Chapter 8 §8.4.4 table).

    Valid augmentations:
    - Time masking: simulates missing sweeps or blink artifacts
    - Frequency masking (≤20 bins): simulates band-limited recordings
    - Gaussian noise injection: simulates different noise floors

    Invalid augmentations (NOT implemented here per ISCEV/Chapter 8):
    - Horizontal flip (time reversal) - creates impossible ERG
    - Vertical flip (frequency inversion) - swaps b-wave and OP bands
    - Random crop/resize - removes critical time-frequency structure
    - Rotation - mixes time and frequency axes
    """
    rng = rng or np.random.default_rng()
    S = S_norm.copy()
    H, W = S.shape

    # Time masking: zero out random block of consecutive time columns
    t_mask = rng.integers(0, max(1, time_mask_max))
    if t_mask > 0:
        t0 = rng.integers(0, max(1, W - t_mask))
        S[:, t0:t0 + t_mask] = 0.0

    # Frequency masking: zero out random block of frequency rows
    f_mask = rng.integers(0, max(1, freq_mask_max))
    if f_mask > 0:
        f0 = rng.integers(0, max(1, H - f_mask))
        S[f0:f0 + f_mask, :] = 0.0

    # Gaussian noise injection
    noise_std = rng.uniform(0.0, noise_std_max)
    if noise_std > 0:
        S = S + rng.normal(0, noise_std, S.shape).astype(np.float32)
        S = np.clip(S, 0.0, 1.0)

    return S


# ============================================================================
# Empirical Window Validation Framework (Chapter 8 §8.3.3)
# ============================================================================

def validate_windows(X_sweeps: np.ndarray, y_labels: np.ndarray,
                     nperseg: int = 64, noverlap: int = 56, fs: float = 2000.0) -> dict:
    """
    Evaluate five window functions on ERG training set and return AUC-ROC table.

    Parameters
    ----------
    X_sweeps : (n_recordings, n_samples) filtered ERG array
    y_labels : (n_recordings,) integer class labels
    nperseg : STFT window size (default 64)
    noverlap : STFT overlap (default 56)
    fs : Sampling rate in Hz (default 2000.0)

    Returns
    -------
    dict : {window_name: mean_auc_roc}

    Chapter 8 Reference: §8.3.3 Empirical Validation Methodology
    """
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import roc_auc_score
    from sklearn.preprocessing import LabelBinarizer
    from sklearn.model_selection import StratifiedKFold

    WINDOWS = ['boxcar', 'triang', 'hann', 'hamming', 'blackman']

    def spectro_features(signal_uv, window):
        _, _, Zxx = stft(signal_uv, fs=fs, window=window,
                         nperseg=nperseg, noverlap=noverlap)
        S = np.abs(Zxx) ** 2
        S_small = resize(S, (64, 64), anti_aliasing=True, preserve_range=True)
        return np.log1p(S_small).ravel()

    lb = LabelBinarizer().fit(y_labels)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    results = {}

    for win in WINDOWS:
        print(f"Evaluating window: {win}...", end=" ", flush=True)
        X_feat = np.array([spectro_features(x, win) for x in X_sweeps])
        fold_aucs = []

        for train_idx, test_idx in cv.split(X_feat, y_labels):
            clf = RandomForestClassifier(n_estimators=100, max_depth=8,
                                         random_state=42, n_jobs=-1)
            clf.fit(X_feat[train_idx], y_labels[train_idx])
            y_prob = clf.predict_proba(X_feat[test_idx])
            auc = roc_auc_score(lb.transform(y_labels[test_idx]), y_prob,
                                multi_class='ovr', average='macro')
            fold_aucs.append(auc)

        mean_auc = float(np.mean(fold_aucs))
        results[win] = mean_auc
        print(f"AUC = {mean_auc:.4f}")

    best = max(results, key=results.get)
    print(f"\nSelected window: {best} (AUC = {results[best]:.4f})")
    return results


print("=" * 60)
print("CELL 7: STFT SPECTROGRAM GENERATION - READY")
print("=" * 60)
print(f"STFT Config: window={STFT_CONFIG['window']}, nperseg={STFT_CONFIG['nperseg']}, "
      f"noverlap={STFT_CONFIG['noverlap']}, fs={STFT_CONFIG['fs']} Hz")
print(f"Output shape: {STFT_CONFIG['output_h']}x{STFT_CONFIG['output_w']}")
print(f"Normalization: {STFT_CONFIG['norm']}")
print("=" * 60)
print("Functions available:")
print("  - generate_erg_spectrogram() : main spectrogram generator")
print("  - flatten_spectrogram_for_ml() : for classical ML input")
print("  - SpectrogramPCAReducer : PCA dimensionality reduction")
print("  - spectrogram_to_tensor() : convert to PyTorch tensor for ViT")
print("  - augment_spectrogram() : data augmentation (time/freq masking, noise)")
print("  - validate_windows() : empirical window selection framework")
print("=" * 60)

CELL 7: STFT SPECTROGRAM GENERATION - READY
STFT Config: window=hamming, nperseg=64, noverlap=56, fs=2000.0 Hz
Output shape: 224x224
Normalization: db_zscore
Functions available:
  - generate_erg_spectrogram() : main spectrogram generator
  - flatten_spectrogram_for_ml() : for classical ML input
  - SpectrogramPCAReducer : PCA dimensionality reduction
  - spectrogram_to_tensor() : convert to PyTorch tensor for ViT
  - augment_spectrogram() : data augmentation (time/freq masking, noise)
  - validate_windows() : empirical window selection framework


In [28]:
# ============================================================================
# CELL 8: SHAP EXPLAINABILITY FOR CLINICAL REPORTS (Chapter 14)
# Implements three-level SHAP framework:
#   Level 1: Feature-level SHAP (TreeExplainer for Random Forest) - FULL
#   Level 2: Spectrogram-level SHAP (GradientExplainer for ViT) - PLACEHOLDER
#   Level 3: Plain-language translation of SHAP values - FULL
# ============================================================================

import numpy as np
import pandas as pd
import warnings
from typing import Dict, Any, List, Tuple, Optional

# Suppress SHAP initialization warnings
warnings.filterwarnings('ignore', category=UserWarning)


class ERGSHAPExplainer:
    """
    SHAP Explainability for ERG Classification Reports
    Implements three-level explainability per Chapter 14:
        Level 1: Feature-level SHAP for Random Forest
        Level 2: Spectrogram-level SHAP for ViT (placeholder for Phase 1)
        Level 3: Plain-language translation of SHAP values
    """

    def __init__(self, config: ERGConfig = None):
        self.config = config or CONFIG
        self._random_forest_model = None
        self._feature_names = None
        self._explainer = None

        # SHAP range to clinical sentence mapping (Chapter 14 §14.4.2)
        self.SHAP_TO_CLINICAL_MAPPING = {
            'a_wave_amplitude': {
                'direction': 'negative',  # Negative Z = reduced amplitude
                'high_importance': 'The a-wave amplitude was severely reduced (strongest contributor)',
                'medium_importance': 'The a-wave amplitude was moderately reduced',
                'low_importance': 'The a-wave amplitude showed mild reduction'
            },
            'a_wave_implicit_time': {
                'direction': 'positive',  # Positive Z = delayed
                'high_importance': 'The a-wave implicit time was severely delayed (strongest contributor)',
                'medium_importance': 'The a-wave implicit time was moderately delayed',
                'low_importance': 'The a-wave implicit time showed mild delay'
            },
            'b_wave_amplitude': {
                'direction': 'negative',
                'high_importance': 'The b-wave amplitude was severely reduced (strongest contributor)',
                'medium_importance': 'The b-wave amplitude was moderately reduced',
                'low_importance': 'The b-wave amplitude showed mild reduction'
            },
            'b_wave_implicit_time': {
                'direction': 'positive',
                'high_importance': 'The b-wave implicit time was severely delayed (strongest contributor)',
                'medium_importance': 'The b-wave implicit time was moderately delayed',
                'low_importance': 'The b-wave implicit time showed mild delay'
            },
            'ba_ratio': {
                'direction': 'negative',
                'high_importance': 'The b/a ratio was severely reduced (strongest contributor)',
                'medium_importance': 'The b/a ratio was moderately reduced',
                'low_importance': 'The b/a ratio showed mild reduction'
            },
            'phnr_amplitude': {
                'direction': 'negative',
                'high_importance': 'The PhNR amplitude was severely reduced (strongest contributor)',
                'medium_importance': 'The PhNR amplitude was moderately reduced',
                'low_importance': 'The PhNR amplitude showed mild reduction'
            }
        }

    def set_random_forest_model(self, model, feature_names: List[str]):
        """
        Set the trained Random Forest model for SHAP explanation.

        Parameters
        ----------
        model : trained RandomForestClassifier
            The Random Forest model from Chapter 12
        feature_names : List[str]
            Names of features used by the model
        """
        self._random_forest_model = model
        self._feature_names = feature_names

        # Initialize TreeExplainer when model is set
        try:
            import shap
            self._explainer = shap.TreeExplainer(model, model_output='probability')
            print(f"✓ SHAP TreeExplainer initialized with {len(feature_names)} features")
        except ImportError:
            print("⚠️ SHAP library not installed. Run: pip install shap")
            self._explainer = None

    def compute_feature_shap(self, feature_vector: np.ndarray) -> np.ndarray:
        """
        Compute SHAP values for a single feature vector.

        Parameters
        ----------
        feature_vector : 1D array of feature values (order must match feature_names)

        Returns
        -------
        shap_values : 1D array of SHAP values for each feature
        """
        if self._explainer is None:
            raise ValueError("Random Forest model not set. Call set_random_forest_model() first.")

        import shap

        # Reshape for single sample
        if feature_vector.ndim == 1:
            feature_vector = feature_vector.reshape(1, -1)

        # Compute SHAP values
        shap_values = self._explainer.shap_values(feature_vector)

        # For binary classification, shap_values[1] is for positive class
        if isinstance(shap_values, list):
            shap_values = shap_values[1]  # Probability of abnormal class

        return shap_values[0]  # Return 1D array for single sample

    def compute_feature_shap_batch(self, feature_matrix: np.ndarray) -> np.ndarray:
        """
        Compute SHAP values for a batch of feature vectors.

        Parameters
        ----------
        feature_matrix : (n_samples, n_features) array

        Returns
        -------
        shap_values : (n_samples, n_features) array of SHAP values
        """
        if self._explainer is None:
            raise ValueError("Random Forest model not set. Call set_random_forest_model() first.")

        import shap

        shap_values = self._explainer.shap_values(feature_matrix)

        # For binary classification, shap_values[1] is for positive class
        if isinstance(shap_values, list):
            shap_values = shap_values[1]

        return shap_values

    def get_top_features(self, shap_values: np.ndarray, top_k: int = 5) -> List[Tuple[str, float]]:
        """
        Return top k features by absolute SHAP value.

        Returns
        -------
        List of (feature_name, shap_value) tuples sorted by importance
        """
        if self._feature_names is None:
            raise ValueError("Feature names not set. Call set_random_forest_model() first.")

        # Get absolute SHAP values and sort
        abs_shap = np.abs(shap_values)
        indices = np.argsort(abs_shap)[::-1][:top_k]

        return [(self._feature_names[i], shap_values[i]) for i in indices if abs_shap[i] > 0]

    def translate_shap_to_text(self, shap_values: np.ndarray,
                                z_scores: Dict[str, float] = None) -> List[str]:
        """
        Convert SHAP values to clinician-readable sentences (Level 3).

        Chapter 14 Reference: §14.4 Plain-Language Translation
        """
        sentences = []

        top_features = self.get_top_features(shap_values, top_k=5)

        for feature_name, shap_val in top_features:
            if abs(shap_val) < 0.01:  # Ignore negligible contributions
                continue

            # Find matching mapping
            mapping = None
            for key, value in self.SHAP_TO_CLINICAL_MAPPING.items():
                if key in feature_name.lower():
                    mapping = value
                    break

            if mapping is None:
                sentences.append(f"{feature_name} contributed {abs(shap_val):.0%} to the classification")
                continue

            # Determine importance level based on absolute SHAP value
            abs_shap_pct = abs(shap_val)
            if abs_shap_pct >= 0.3:
                importance = 'high_importance'
            elif abs_shap_pct >= 0.15:
                importance = 'medium_importance'
            else:
                importance = 'low_importance'

            # Get direction sign for proper clinical wording
            direction = mapping['direction']
            shap_sign = np.sign(shap_val)

            # Adjust wording based on sign
            if (direction == 'negative' and shap_sign < 0) or (direction == 'positive' and shap_sign > 0):
                sentence = mapping[importance]
            else:
                # Opposing direction (e.g., increased amplitude when reduction is typical)
                if 'amplitude' in feature_name:
                    sentence = f"The {feature_name.replace('_', ' ')} showed paradoxical increase (opposite of typical pattern)"
                else:
                    sentence = f"The {feature_name.replace('_', ' ')} showed atypical timing"

            sentences.append(f"{sentence} (contribution: {abs_shap_pct:.0%})")

        return sentences

    def generate_shap_report(self, feature_vector: np.ndarray,
                            z_scores: Dict[str, float] = None,
                            top_k: int = 5) -> Dict[str, Any]:
        """
        Generate complete SHAP report for a single recording.

        Returns
        -------
        Dictionary with:
            - shap_values: raw SHAP values array
            - top_features: list of (feature, value) for top_k features
            - plain_language_sentences: list of clinician-readable sentences
            - waterfall_data: formatted for plotting
        """
        if self._explainer is None:
            return {
                'status': 'UNAVAILABLE',
                'message': 'SHAP explainer not initialized. Train Random Forest model first.',
                'shap_values': None,
                'top_features': [],
                'plain_language_sentences': [],
                'waterfall_data': None
            }

        shap_values = self.compute_feature_shap(feature_vector)
        top_features = self.get_top_features(shap_values, top_k)
        sentences = self.translate_shap_to_text(shap_values, z_scores)

        # Prepare waterfall data for visualization
        waterfall_data = []
        for name, val in top_features:
            waterfall_data.append({
                'feature': name.replace('_', ' ').title(),
                'shap_value': float(val),
                'abs_shap': float(abs(val)),
                'direction': 'positive' if val > 0 else 'negative'
            })

        return {
            'status': 'AVAILABLE',
            'message': 'SHAP explanation generated successfully',
            'shap_values': shap_values.tolist() if shap_values is not None else None,
            'top_features': [(name, float(val)) for name, val in top_features],
            'plain_language_sentences': sentences,
            'waterfall_data': waterfall_data
        }

    # ========================================================================
    # Level 2: Spectrogram-level SHAP (Placeholder for Phase 1)
    # Chapter 14 Reference: §14.3 Spectrogram-Level SHAP
    # ========================================================================

    def get_spectrogram_shap_placeholder(self) -> Dict[str, Any]:
        """
        Placeholder for ViT GradientExplainer (requires trained ViT model).
        Returns structured placeholder for Layer 3 Specialist Report.
        """
        return {
            'status': 'PENDING',
            'message': 'Spectrogram-level SHAP (GradientExplainer) requires trained Vision Transformer (Chapter 13)',
            'implementation_required': [
                'Train ViT model on STFT spectrograms (Chapter 13)',
                'Integrate shap.GradientExplainer for ViT',
                'Generate saliency overlays on time-frequency spectrograms'
            ],
            'saliency_overlay': None,
            'attention_maps': None
        }


# ============================================================================
# SHAP Integration with Layer 3 Specialist Report
# ============================================================================

def integrate_shap_into_report(full_report: Dict, shap_report: Dict) -> Dict:
    """
    Integrate SHAP report into the Layer 3 Specialist Report structure.
    """
    if 'layer_3_specialist' not in full_report:
        full_report['layer_3_specialist'] = {}

    full_report['layer_3_specialist']['shap_explainability'] = {
        'level_1_feature_shap': shap_report,
        'level_2_spectrogram_shap': {
            'status': 'PENDING',
            'message': 'Requires trained Vision Transformer (Chapter 13)'
        },
        'level_3_plain_language': shap_report.get('plain_language_sentences', [])
    }

    return full_report


# ============================================================================
# Example: SHAP Waterfall Plot Generation (Optional)
# ============================================================================

def plot_shap_waterfall(shap_values: np.ndarray, feature_names: List[str],
                        feature_values: np.ndarray = None,
                        max_display: int = 10):
    """
    Generate SHAP waterfall plot for a single prediction.

    Requires matplotlib and shap libraries.
    This is a wrapper for shap.plots.waterfall().
    """
    try:
        import shap
        import matplotlib.pyplot as plt

        # Create Explanation object
        expected_value = 0.0  # Will be set properly when model is available
        explanation = shap.Explanation(
            values=shap_values,
            base_values=expected_value,
            data=feature_values if feature_values is not None else np.zeros_like(shap_values),
            feature_names=feature_names
        )

        # Generate waterfall plot
        shap.plots.waterfall(explanation, max_display=max_display, show=False)
        plt.title("SHAP Waterfall Plot: Feature Contributions")
        plt.tight_layout()

        return plt.gcf()

    except ImportError:
        print("⚠️ SHAP or matplotlib not available for plotting")
        return None
    except Exception as e:
        print(f"⚠️ Could not generate waterfall plot: {e}")
        return None


# ============================================================================
# Demonstration
# ============================================================================

print("\n" + "=" * 60)
print("CELL 8: SHAP EXPLAINABILITY - READY")
print("=" * 60)
print("Three-Level SHAP Framework (Chapter 14):")
print("  Level 1: Feature-level SHAP (TreeExplainer for Random Forest) - ✅ FULL")
print("  Level 2: Spectrogram-level SHAP (GradientExplainer for ViT) - ⏳ PLACEHOLDER")
print("  Level 3: Plain-language translation - ✅ FULL")
print("=" * 60)
print("\nUsage Example:")
print("  explainer = ERGSHAPExplainer()")
print("  explainer.set_random_forest_model(rf_model, feature_names)")
print("  shap_report = explainer.generate_shap_report(feature_vector, z_scores)")
print("  full_report = integrate_shap_into_report(full_report, shap_report)")
print("=" * 60)
print("\n⚠️ Note: Full SHAP functionality requires:")
print("  1. Trained Random Forest model (Chapter 12)")
print("  2. SHAP library installed: pip install shap")
print("  3. Feature names matching training data")
print("=" * 60)


CELL 8: SHAP EXPLAINABILITY - READY
Three-Level SHAP Framework (Chapter 14):
  Level 1: Feature-level SHAP (TreeExplainer for Random Forest) - ✅ FULL
  Level 2: Spectrogram-level SHAP (GradientExplainer for ViT) - ⏳ PLACEHOLDER
  Level 3: Plain-language translation - ✅ FULL

Usage Example:
  explainer = ERGSHAPExplainer()
  explainer.set_random_forest_model(rf_model, feature_names)
  shap_report = explainer.generate_shap_report(feature_vector, z_scores)
  full_report = integrate_shap_into_report(full_report, shap_report)

⚠️ Note: Full SHAP functionality requires:
  1. Trained Random Forest model (Chapter 12)
  2. SHAP library installed: pip install shap
  3. Feature names matching training data


In [29]:
# ============================================================================
# CELL 9: FHIR JSON OUTPUT (HL7 FHIR R4 Compliant)
# Implements Appendix B: Complete FHIR Observation Resource for ERG
# ============================================================================

from datetime import datetime
from typing import Dict, Any, List, Optional
import json


class ERGFHIRGenerator:
    """
    HL7 FHIR R4 compliant JSON generator for ERG reports.
    Implements Appendix B: Observation Resource Template for ERG Waveform

    FHIR Resources:
    - Observation (LOINC 26456-9: Electroretinogram)
    - Component per ERG parameter (a-wave, b-wave, OPs, PhNR, flicker)
    - Interpretation codes: N (normal), A (borderline), HH (abnormal)
    """

    # LOINC codes for ERG components (Appendix B)
    LOINC_CODES = {
        'a_wave_amplitude': '93867-5',
        'a_wave_implicit_time': '93868-3',
        'b_wave_amplitude': '93869-1',
        'b_wave_implicit_time': '93870-9',
        'ba_ratio': '93871-7',
        'op2_amplitude': '93872-5',
        'op3_amplitude': '93873-3',
        'op4_amplitude': '93874-1',
        'op_sum': '93875-8',
        'phnr_amplitude': '93876-6',
        'flicker_amplitude': '93877-4',
        'flicker_implicit_time': '93878-2'
    }

    # SNOMED CT codes for interpretation
    SNOMED_INTERPRETATION = {
        'GREEN': {'code': '17621005', 'display': 'Normal'},
        'AMBER': {'code': '263654007', 'display': 'Borderline'},
        'RED': {'code': '442257004', 'display': 'Abnormal'},
        'YELLOW': {'code': '418186005', 'display': 'Inconclusive'}
    }

    # ISCEV protocol codes (custom extension)
    ISCEV_PROTOCOL_CODES = {
        'DA 0.01': 'ISCEV-DA-001',
        'DA 3': 'ISCEV-DA-300',
        'DA 10': 'ISCEV-DA-1000',
        'LA 3': 'ISCEV-LA-300',
        'LA 30 Hz': 'ISCEV-LA-30HZ'
    }

    def __init__(self, config: ERGConfig = None):
        self.config = config or CONFIG

    def generate_observation(self,
                            report_id: str,
                            patient_id: str,
                            traffic_light: Dict,
                            features: Dict,
                            z_scores: Dict,
                            audit_results: Dict,
                            electrode_type: str,
                            protocol: str) -> Dict[str, Any]:
        """
        Generate FHIR Observation resource for ERG.

        Parameters
        ----------
        report_id : Unique identifier for this report
        patient_id : Anonymized patient reference
        traffic_light : Traffic light dictionary from Cell 6
        features : Extracted features dictionary
        z_scores : Z-scores dictionary
        audit_results : Audit results from Stage 1
        electrode_type : Electrode type string
        protocol : ISCEV protocol string

        Returns
        -------
        Observation resource as dictionary (FHIR R4 compliant)
        """

        # Base Observation resource
        observation = {
            "resourceType": "Observation",
            "id": report_id,
            "meta": {
                "profile": ["http://hl7.org/fhir/StructureDefinition/Observation"],
                "versionId": "1",
                "lastUpdated": datetime.now().isoformat()
            },
            "status": "final",
            "category": [
                {
                    "coding": [
                        {
                            "system": "http://terminology.hl7.org/CodeSystem/observation-category",
                            "code": "exam",
                            "display": "Exam"
                        }
                    ]
                }
            ],
            "code": {
                "coding": [
                    {
                        "system": "http://loinc.org",
                        "code": "26456-9",
                        "display": "Electroretinogram (ERG)"
                    }
                ],
                "text": f"Full-field ERG - {protocol}"
            },
            "subject": {
                "reference": f"Patient/{patient_id}"
            },
            "effectiveDateTime": datetime.now().isoformat(),
            "issued": datetime.now().isoformat(),
            "performer": [
                {
                    "reference": "Organization/ERG-Processing-API",
                    "display": "ERG Processing Pipeline v2.1.0"
                }
            ],
            "valueCodeableConcept": {
                "coding": [
                    {
                        "system": "http://snomed.info/sct",
                        "code": self.SNOMED_INTERPRETATION[traffic_light['signal']]['code'],
                        "display": self.SNOMED_INTERPRETATION[traffic_light['signal']]['display']
                    }
                ],
                "text": traffic_light['message']
            },
            "interpretation": [
                {
                    "coding": [
                        {
                            "system": "http://terminology.hl7.org/CodeSystem/v3-ObservationInterpretation",
                            "code": "N" if traffic_light['signal'] == 'GREEN' else ("A" if traffic_light['signal'] == 'AMBER' else "HH"),
                            "display": traffic_light['signal']
                        }
                    ]
                }
            ],
            "method": {
                "coding": [
                    {
                        "system": "https://iscev.org/standards",
                        "code": self.ISCEV_PROTOCOL_CODES.get(protocol, "ISCEV-UNKNOWN"),
                        "display": f"ISCEV 2022 {protocol}"
                    }
                ]
            },
            "device": {
                "coding": [
                    {
                        "system": "https://github.com/ERG-AI/pipeline",
                        "code": self.config.PIPELINE_VERSION,
                        "display": self.config.PIPELINE_NAME
                    }
                ]
            },
            "component": []
        }

        # Add quality component
        observation['component'].append({
            "code": {
                "coding": [
                    {
                        "system": "http://loinc.org",
                        "code": "93879-0",
                        "display": "Signal quality grade"
                    }
                ]
            },
            "valueCodeableConcept": {
                "coding": [
                    {
                        "system": "https://github.com/ERG-AI/quality",
                        "code": audit_results.get('quality', {}).get('grade', 'U'),
                        "display": audit_results.get('quality', {}).get('description', 'Unknown')
                    }
                ]
            }
        })

        # Add electrode type component
        observation['component'].append({
            "code": {
                "coding": [
                    {
                        "system": "https://iscev.org/electrodes",
                        "code": electrode_type.upper(),
                        "display": f"{electrode_type.replace('_', ' ').title()} electrode"
                    }
                ]
            },
            "valueString": electrode_type.replace('_', ' ')
        })

        # Add components for each feature with Z-score
        for param_name, z_value in z_scores.items():
            if abs(z_value) > 0:
                # Map parameter to LOINC code
                loinc_code = self.LOINC_CODES.get(param_name, None)
                param_display = param_name.replace('_', ' ').title()

                component = {
                    "code": {
                        "text": param_display
                    },
                    "valueQuantity": {
                        "value": z_value,
                        "unit": "Z-score",
                        "system": "http://unitsofmeasure.org",
                        "code": "{z-score}"
                    },
                    "interpretation": [
                        {
                            "coding": [
                                {
                                    "system": "http://terminology.hl7.org/CodeSystem/v3-ObservationInterpretation",
                                    "code": "N" if abs(z_value) <= 2 else ("A" if abs(z_value) <= 3 else "HH"),
                                    "display": "Normal" if abs(z_value) <= 2 else ("Borderline" if abs(z_value) <= 3 else "Abnormal")
                                }
                            ]
                        }
                    ]
                }

                if loinc_code:
                    component["code"]["coding"] = [{
                        "system": "http://loinc.org",
                        "code": loinc_code,
                        "display": param_display
                    }]
                else:
                    component["code"]["text"] = param_display

                observation['component'].append(component)

        return observation

    def generate_bundle(self, observation: Dict, include_patient: bool = False) -> Dict[str, Any]:
        """
        Wrap Observation in a FHIR Bundle.

        Parameters
        ----------
        observation : Observation resource dictionary
        include_patient : If True, include Patient resource (requires full patient data)

        Returns
        -------
        Bundle resource as dictionary
        """
        bundle = {
            "resourceType": "Bundle",
            "type": "collection",
            "id": f"bundle-{observation['id']}",
            "meta": {
                "lastUpdated": datetime.now().isoformat()
            },
            "entry": [
                {
                    "fullUrl": f"urn:uuid:{observation['id']}",
                    "resource": observation
                }
            ]
        }

        return bundle

    def to_json(self, fhir_resource: Dict, pretty: bool = True) -> str:
        """
        Convert FHIR resource to JSON string.

        Parameters
        ----------
        fhir_resource : FHIR resource dictionary
        pretty : If True, format with indentation

        Returns
        -------
        JSON string
        """
        if pretty:
            return json.dumps(fhir_resource, indent=2, default=str)
        return json.dumps(fhir_resource, default=str)

    def save_to_file(self, fhir_resource: Dict, filename: str) -> None:
        """
        Save FHIR resource to JSON file.

        Parameters
        ----------
        fhir_resource : FHIR resource dictionary
        filename : Output filename (should end with .json)
        """
        with open(filename, 'w') as f:
            json.dump(fhir_resource, f, indent=2, default=str)
        print(f"✓ FHIR JSON saved to: {filename}")


# ============================================================================
# Integration with Cell 6 Report
# ============================================================================

def add_fhir_to_report(full_report: Dict, fhir_observation: Dict) -> Dict:
    """
    Add FHIR Observation to the full report dictionary.
    """
    full_report['fhir_observation'] = fhir_observation
    full_report['fhir_bundle'] = None  # Will be generated if needed
    return full_report


# ============================================================================
# Example Usage
# ============================================================================

print("\n" + "=" * 60)
print("CELL 9: FHIR JSON OUTPUT - READY")
print("=" * 60)
print("HL7 FHIR R4 Compliant ERG Observation Resource")
print("LOINC Codes: 26456-9 (ERG), plus component codes for all parameters")
print("SNOMED CT: Normal (17621005), Borderline (263654007), Abnormal (442257004)")
print("ISCEV Protocol Extensions: Custom codes for 5 standard protocols")
print("=" * 60)
print("\nUsage Example:")
print("  fhir_gen = ERGFHIRGenerator()")
print("  observation = fhir_gen.generate_observation(")
print("      report_id='ERG-20260514-120000',")
print("      patient_id='P001',")
print("      traffic_light=tl_dict,")
print("      features=features_dict,")
print("      z_scores=z_scores_dict,")
print("      audit_results=audit_dict,")
print("      electrode_type='contact_lens',")
print("      protocol='DA 3'")
print("  )")
print("  fhir_json = fhir_gen.to_json(observation)")
print("  fhir_gen.save_to_file(observation, 'fhir_observation.json')")
print("=" * 60)


CELL 9: FHIR JSON OUTPUT - READY
HL7 FHIR R4 Compliant ERG Observation Resource
LOINC Codes: 26456-9 (ERG), plus component codes for all parameters
SNOMED CT: Normal (17621005), Borderline (263654007), Abnormal (442257004)
ISCEV Protocol Extensions: Custom codes for 5 standard protocols

Usage Example:
  fhir_gen = ERGFHIRGenerator()
  observation = fhir_gen.generate_observation(
      report_id='ERG-20260514-120000',
      patient_id='P001',
      traffic_light=tl_dict,
      features=features_dict,
      z_scores=z_scores_dict,
      audit_results=audit_dict,
      electrode_type='contact_lens',
      protocol='DA 3'
  )
  fhir_json = fhir_gen.to_json(observation)
  fhir_gen.save_to_file(observation, 'fhir_observation.json')


In [30]:
# ============================================================================
# CELL 10: COLAB UI INTEGRATION - FULL PIPELINE EXECUTION
# Place this cell at the VERY END of your notebook
# ============================================================================

import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files
import time

print("=" * 60)
print("ERG PROCESSING PIPELINE - CLINICAL DECISION SUPPORT API")
print(f"ISCEV 2022 Compliant | Version {CONFIG.PIPELINE_VERSION}")
print("=" * 60)

# File upload
upload_widget = widgets.FileUpload(accept='.csv', multiple=False, description='Upload ERG File', button_style='primary')

# Patient inputs
age_group_widget = widgets.Dropdown(
    options=['18-80y', '13-17y', '6-12y', '1-5y', '0-12mo', '80+y', 'unknown'],
    value='18-80y',
    description='Age Group:'
)
pupil_status_widget = widgets.RadioButtons(
    options=['Dilated (≥6mm)', 'Natural'],
    value='Dilated (≥6mm)',
    description='Pupil Status:'
)
pupil_diameter_widget = widgets.FloatSlider(
    value=6.0, min=0, max=10, step=0.5,
    description='Pupil Diameter (mm):'
)
electrode_widget = widgets.Dropdown(
    options=['contact_lens', 'gold_foil', 'dtl_fiber', 'skin'],
    value='contact_lens',
    description='Electrode Type:'
)
protocol_widget = widgets.ToggleButtons(
    options=['DA 0.01', 'DA 3', 'DA 10', 'LA 3', 'LA 30 Hz'],
    value='DA 3',
    description='Protocol:'
)
environment_widget = widgets.Dropdown(
    options=['Shielded lab', 'Clinic', 'Portable'],
    value='Shielded lab',
    description='Environment:'
)
flash_duration_widget = widgets.FloatText(
    value=1.0,
    description='Flash Duration (ms):',
    help='For LED stimulators: ≥5 ms enables midpoint correction'
)

# Pre-stimulus baseline duration widget
pre_stimulus_widget = widgets.FloatText(
    value=50.0,
    description='Pre-stimulus (ms):',
    help='Duration before flash (ISCEV minimum 20 ms)',
    style={'description_width': 'initial'}
)

# OP extraction default ON for DA 3.0 and DA 10.0 protocols
op_extract_widget = widgets.Checkbox(value=True, description='Extract Oscillatory Potentials (75-300 Hz)')

# Dynamic update function for OP extraction default
def update_op_default(change):
    """Set OP extraction default based on protocol"""
    protocol = protocol_widget.value
    if protocol in ['DA 3', 'DA 10']:
        op_extract_widget.value = True
    else:
        op_extract_widget.value = False

protocol_widget.observe(update_op_default, names='value')

# Notch filter widgets
notch_override_widget = widgets.Checkbox(
    value=False,
    description='⚠️ OVERRIDE: Apply Notch Filter (ISCEV advises against)'
)
notch_consent_widget = widgets.Checkbox(
    value=False,
    description='I confirm clinical justification for notch filter override'
)

# Run button
run_button = widgets.Button(
    description='▶ RUN PIPELINE',
    button_style='success',
    layout=widgets.Layout(width='200px')
)
output_area = widgets.Output()

# Layout
ui = widgets.VBox([
    upload_widget,
    widgets.HTML("<hr>"),
    widgets.HTML("<b>📋 PATIENT INFORMATION</b>"),
    age_group_widget,
    widgets.HBox([pupil_status_widget, pupil_diameter_widget]),
    electrode_widget,
    environment_widget,
    widgets.HTML("<hr>"),
    widgets.HTML("<b>⚡ PROTOCOL & RECORDING</b>"),
    protocol_widget,
    flash_duration_widget,
    pre_stimulus_widget,
    op_extract_widget,
    widgets.HTML("<hr>"),
    widgets.HTML("<b>⚠️ ISCEV 2022 COMPLIANCE</b>"),
    widgets.HTML("<font color='red'>Notch filters are NOT recommended by ISCEV (Page 7, Col 2, Para 4)</font>"),
    notch_override_widget,
    notch_consent_widget,
    widgets.HTML("<hr>"),
    run_button,
    output_area
])

display(ui)

def run_pipeline(b):
    with output_area:
        clear_output()

        # Start timing
        t_start = time.perf_counter()

        if not upload_widget.value:
            print("❌ Please upload an ERG CSV file first")
            return

        if notch_override_widget.value and not notch_consent_widget.value:
            print("❌ Notch filter override requires confirmation of clinical justification")
            return

        uploaded_file = list(upload_widget.value.values())[0]
        filename = uploaded_file['metadata']['name']
        content = uploaded_file['content']

        temp_path = f"/tmp/{filename}"
        with open(temp_path, 'wb') as f:
            f.write(content)

        print("=" * 60)
        print("PROCESSING ERG SIGNAL...")
        print("=" * 60)
        print(f"File: {filename}")
        print(f"Age Group: {age_group_widget.value}")
        print(f"Electrode: {electrode_widget.value}")
        print(f"Protocol: {protocol_widget.value}")
        print(f"Flash Duration: {flash_duration_widget.value} ms")
        print(f"Pre-stimulus Duration: {pre_stimulus_widget.value} ms")
        print(f"Notch Filter: {'APPLIED (override)' if notch_override_widget.value else 'SKIPPED (ISCEV compliant)'}")
        print("-" * 60)

        try:
            # Load CSV file
            time_ms, signal_uv, fs_hz, metadata = load_erg_csv(temp_path)
            print(f"✓ Signal loaded: {len(signal_uv)} samples @ {fs_hz:.0f} Hz")

            # Calculate flash onset sample
            pre_stimulus_ms = pre_stimulus_widget.value
            flash_onset_sample = int(pre_stimulus_ms * fs_hz / 1000)
            prestimulus_samples = flash_onset_sample

            # Calculate noise RMS
            if prestimulus_samples > 0 and prestimulus_samples < len(signal_uv):
                pre_stim_signal = signal_uv[:prestimulus_samples]
                noise_rms_uv = float(np.sqrt(np.mean(pre_stim_signal**2)))
            else:
                noise_rms_uv = 1.0
                print(f"⚠️ Warning: Could not extract pre-stimulus baseline. Using default noise RMS.")

            # Run audit
            auditor = ERGAudit()
            audit_result = auditor.run_full_audit(
                signal_uv, fs_hz,
                electrode_type=electrode_widget.value,
                prestimulus_samples=prestimulus_samples,
                age_group=age_group_widget.value
            )

            print(f"✓ Audit complete: Grade {audit_result['quality']['grade']}")
            print(f"  OP Available: {audit_result['oscillatory_potentials']['available']}")
            print(f"  SNR: {audit_result['snr']['db']} dB ({audit_result['snr']['status']})")

            # Run filter pipeline
            filter_obj = ERGFilter()
            hardware_cutoff = audit_result['bandwidth'].get('hardware_cutoff_hz')

            filtered_signal, filter_log = filter_obj.run_filter_pipeline(
                signal_uv, fs_hz,
                apply_notch=notch_override_widget.value,
                notch_hz=50.0,
                hardware_cutoff_hz=hardware_cutoff,
                user_confirmed_notch=notch_consent_widget.value
            )

            print(f"✓ Filtering complete")

            # Extract features
            extractor = ERGFeatureExtractor()
            op_signal = None
            if op_extract_widget.value and audit_result['oscillatory_potentials']['available']:
                op_signal = filter_obj.extract_ops(filtered_signal, fs_hz)
                print(f"✓ OP signal extracted (75-300 Hz)")

            # Extract all features
            features = extractor.extract_all_features(
                signal=filtered_signal,
                fs_hz=fs_hz,
                protocol=protocol_widget.value,
                flash_onset_sample=flash_onset_sample,
                flash_duration_ms=flash_duration_widget.value,
                op_signal=op_signal,
                noise_rms_uv=noise_rms_uv
            )

            print(f"✓ Features extracted")
            if 'a_wave_amplitude_uv' in features and not np.isnan(features['a_wave_amplitude_uv']):
                print(f"  a-wave: {features['a_wave_amplitude_uv']} µV @ {features['a_wave_implicit_time_ms']} ms")
                print(f"  b-wave: {features['b_wave_amplitude_uv']} µV @ {features['b_wave_implicit_time_ms']} ms")
            if 'phnr_amp_uv' in features and not np.isnan(features['phnr_amp_uv']):
                print(f"  PhNR: {features['phnr_amp_uv']} µV")
            if 'oscillatory_potentials' in features:
                ops = features['oscillatory_potentials']
                print(f"  OPs: OP2={ops.get('OP2',0)} µV, OP3={ops.get('OP3',0)} µV, OP4={ops.get('OP4',0)} µV")

            # Calculate processing time
            processing_time_ms = (time.perf_counter() - t_start) * 1000

            # Generate full report
            report_gen = ERGReportGenerator()
            electrode_value = electrode_widget.value
            if hasattr(electrode_value, 'value'):
                electrode_value = electrode_value.value

            full_report = report_gen.generate_full_report(
                features=features,
                audit_results=audit_result,
                electrode_type=electrode_value,
                protocol=protocol_widget.value,
                filtered_signal=filtered_signal,
                time_ms=time_ms,
                filter_log=filter_log,
                processing_time_ms=processing_time_ms
            )

            # Generate FHIR JSON (Cell 9)
            fhir_gen = ERGFHIRGenerator()
            fhir_observation = fhir_gen.generate_observation(
                report_id=full_report['layer_4_technical_audit']['report_id'],
                patient_id="P001",  # In production, get from user input
                traffic_light=full_report['layer_1_traffic_light'],
                features=features,
                z_scores=full_report['layer_2_clinical_summary']['z_scores'],
                audit_results=audit_result,
                electrode_type=electrode_value,
                protocol=protocol_widget.value
            )
            full_report['fhir_observation'] = fhir_observation

            print("-" * 60)
            print("\n" + "=" * 60)
            print("FINAL REPORT")
            print("=" * 60)

            tl = full_report['layer_1_traffic_light']
            print(f"\n{tl['color']} TRAFFIC LIGHT: {tl['signal']}")
            print(f"   {tl['message']}")
            print(f"   Confidence: {tl['confidence']:.0%}")
            print(f"\n📊 QUALITY GRADE: {audit_result['quality']['grade']}")
            print(f"   {audit_result['quality']['description']}")
            print(f"\n📋 RECOMMENDED ACTION: {full_report['layer_2_clinical_summary']['recommended_action']}")
            print(f"\n⏱️ Processing time: {processing_time_ms:.0f} ms")
            print("\n" + "=" * 60)
            print("✅ Pipeline execution complete")
            print("=" * 60)

        except Exception as e:
            print(f"❌ Error: {e}")
            import traceback
            traceback.print_exc()

run_button.on_click(run_pipeline)
print("\n✓ UI Ready - Please upload an ERG CSV file and click 'RUN PIPELINE'")

ERG PROCESSING PIPELINE - CLINICAL DECISION SUPPORT API
ISCEV 2022 Compliant | Version 2.3.2



✓ UI Ready - Please upload an ERG CSV file and click 'RUN PIPELINE'
